In [1]:
import os, sys

MODULE_BASE = "/mnt/c/Users/JatinJangid/.vscode/adtech-databricks/AdTech - Finance project/Adtech - generators/generator_src"

for subdir in ["core", "simulation", "generators", "edge_cases", "validation"]:
    full_path = f"{MODULE_BASE}/{subdir}"
    os.makedirs(full_path, exist_ok=True)
    init_path = f"{full_path}/__init__.py"
    if not os.path.exists(init_path):
        open(init_path, "w").close()

open(f"{MODULE_BASE}/__init__.py", "w").close()

if MODULE_BASE not in sys.path:
    sys.path.insert(0, MODULE_BASE)

print(f"Directories ready. MODULE_BASE = {MODULE_BASE}")
print(f"sys.path[0] = {sys.path[0]}")

Directories ready. MODULE_BASE = /mnt/c/Users/JatinJangid/.vscode/adtech-databricks/AdTech - Finance project/Adtech - generators/generator_src
sys.path[0] = /mnt/c/Users/JatinJangid/.vscode/adtech-databricks/AdTech - Finance project/Adtech - generators/generator_src


In [2]:
code = """
import os
import re
import yaml
import json
import logging
from pathlib import Path
from typing import Any, Dict, Optional
from functools import lru_cache
from copy import deepcopy

logger = logging.getLogger(__name__)


class ConfigurationError(Exception):
    pass


class DotDict:

    def __init__(self, data: Dict):
        self._data = data

    def __getattr__(self, key: str) -> Any:
        if key.startswith("_"):
            return super().__getattribute__(key)
        try:
            val = self._data[key]
            return DotDict(val) if isinstance(val, dict) else val
        except KeyError:
            raise AttributeError(f"Config key '{key}' not found")

    def __getitem__(self, key: str) -> Any:
        val = self._data[key]
        return DotDict(val) if isinstance(val, dict) else val

    def get(self, key: str, default: Any = None) -> Any:
        val = self._data.get(key, default)
        return DotDict(val) if isinstance(val, dict) else val

    def to_dict(self) -> Dict:
        return deepcopy(self._data)

    def __contains__(self, key: str) -> bool:
        return key in self._data

    def __repr__(self) -> str:
        return f"DotDict({self._data})"


class ConfigLoader:

    ENV_VAR_PATTERN = re.compile(r"\$\{([^}]+)\}")

    def __init__(self, config_path: str, overrides: Optional[Dict] = None):
        self._path = config_path
        self._overrides = overrides or {}
        self._raw: Dict = {}
        self._config: Optional[DotDict] = None

    def load(self) -> "ConfigLoader":
        try:
            with open(self._path, "r") as f:
                raw = yaml.safe_load(f.read())
        except FileNotFoundError:
            raise ConfigurationError(f"Config file not found: {self._path}")
        except yaml.YAMLError as e:
            raise ConfigurationError(f"Invalid YAML in {self._path}: {e}")

        raw = self._interpolate_env_vars(raw)
        raw = self._deep_merge(raw, self._overrides)
        self._raw = raw
        self._config = DotDict(raw)
        logger.info(f"Configuration loaded from {self._path}")
        return self

    def _interpolate_env_vars(self, obj: Any) -> Any:
        if isinstance(obj, str):
            def replace(match):
                var_name = match.group(1)
                value = os.environ.get(var_name)
                if value is None:
                    logger.warning(f"Environment variable ${{{var_name}}} not set")
                return value or match.group(0)
            return self.ENV_VAR_PATTERN.sub(replace, obj)
        elif isinstance(obj, dict):
            return {k: self._interpolate_env_vars(v) for k, v in obj.items()}
        elif isinstance(obj, list):
            return [self._interpolate_env_vars(i) for i in obj]
        return obj

    def _deep_merge(self, base: Dict, override: Dict) -> Dict:
        result = deepcopy(base)
        for key, value in override.items():
            if key in result and isinstance(result[key], dict) and isinstance(value, dict):
                result[key] = self._deep_merge(result[key], value)
            else:
                result[key] = value
        return result

    @property
    def config(self) -> DotDict:
        if self._config is None:
            raise ConfigurationError("Config not loaded. Call .load() first.")
        return self._config

    def section(self, key: str) -> DotDict:
        return self.config[key]
"""
path = "/mnt/c/Users/JatinJangid/.vscode/adtech-databricks/AdTech - Finance project/Adtech - generators/generator_src/core/config_loader.py"

with open(path, "w") as f:
    f.write(code.strip())
print(f"Written: {path}")

Written: /mnt/c/Users/JatinJangid/.vscode/adtech-databricks/AdTech - Finance project/Adtech - generators/generator_src/core/config_loader.py


<>:53: SyntaxWarning: "\$" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\$"? A raw string is also an option.
<>:53: SyntaxWarning: "\$" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\$"? A raw string is also an option.
/tmp/ipykernel_753/1993742402.py:53: SyntaxWarning: "\$" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\$"? A raw string is also an option.
  ENV_VAR_PATTERN = re.compile(r"\$\{([^}]+)\}")


In [3]:
code = """
import abc
import json
import logging
import os
import time
import uuid
import hashlib
from datetime import datetime, timezone, timedelta
from pathlib import Path
from typing import Any, Dict, List, Optional, Tuple
import numpy as np

logger = logging.getLogger(__name__)


class GeneratorMetrics:

    def __init__(self, generator_name: str):
        self.name = generator_name
        self.records_generated = 0
        self.records_written = 0
        self.records_failed_validation = 0
        self.fraud_injected = 0
        self.late_events_injected = 0
        self.duplicates_injected = 0
        self.write_errors = 0
        self._start_time = time.monotonic()

    def throughput_rps(self) -> float:
        elapsed = time.monotonic() - self._start_time
        return self.records_written / elapsed if elapsed > 0 else 0.0

    def to_dict(self) -> Dict:
        return {
            "generator": self.name,
            "records_generated": self.records_generated,
            "records_written": self.records_written,
            "records_failed_validation": self.records_failed_validation,
            "fraud_injected": self.fraud_injected,
            "late_events_injected": self.late_events_injected,
            "duplicates_injected": self.duplicates_injected,
            "write_errors": self.write_errors,
            "throughput_rps": round(self.throughput_rps(), 2),
        }

    def log(self):
        logger.info(f"[METRICS] {json.dumps(self.to_dict())}")


class BaseEventGenerator(abc.ABC):
    # Subclasses declare their event_type
    EVENT_TYPE: str = None

    def __init__(
        self,
        config,             # DotDict from ConfigLoader
        shared_state,       # SharedGeneratorState (injected)
        edge_injectors,     # List[BaseEdgeCaseInjector]
        validator=None,     # Optional SchemaValidator
    ):
        assert self.EVENT_TYPE is not None, "Subclass must define EVENT_TYPE"
        self.config = config
        self.shared_state = shared_state
        self.edge_injectors = edge_injectors
        self.validator = validator

        # Deterministic RNG: each generator type gets a unique seed derived
        # from the global seed + event type hash. This ensures reproducibility
        # while maintaining statistical independence between generators.
        base_seed = config.platform.seed
        type_hash = int(hashlib.md5(self.EVENT_TYPE.encode()).hexdigest()[:8], 16)
        self._rng = np.random.default_rng(base_seed ^ type_hash)

        self.metrics = GeneratorMetrics(self.EVENT_TYPE)
        self._output_base = config.platform.output_base_path
        self._batch_interval = config.platform.batch_interval_seconds
        self._rate_cfg = config.event_rates[self.EVENT_TYPE.replace("-", "_")]
        self._validate_enabled = config.platform.enable_validation

        # Dedup buffer: circular buffer of recent event_ids for duplicate injection
        self._recent_event_ids: List[str] = []
        self._dedup_buffer_size = 1000

        # Ensure output directory exists
        self._output_dir = self._build_output_path()
        os.makedirs(self._output_dir, exist_ok=True)

    def _build_output_path(self) -> str:
        now = datetime.now(timezone.utc)
        return os.path.join(
            self._output_base,
            self.EVENT_TYPE,
            f"dt={now.strftime('%Y-%m-%d')}",
            f"hr={now.strftime('%H')}",
        )

    def generate_event_id(self) -> str:
        return str(uuid.uuid4())

    def current_event_time(self) -> datetime:

        if self.config.platform.replay_mode:
            base = datetime.fromisoformat(self.config.platform.replay_timestamp)
            elapsed = timedelta(seconds=time.monotonic())
            return base + elapsed
        return datetime.now(timezone.utc)

    def to_iso(self, dt: datetime) -> str:
        return dt.strftime("%Y-%m-%dT%H:%M:%S.%f")[:-3] + "Z"

    @abc.abstractmethod
    def _generate_record(self) -> Dict[str, Any]:

        pass

    @abc.abstractmethod
    def _get_schema(self) -> Dict:
        pass

    def _calculate_batch_size(self) -> int:

        traffic_cfg = self.config.traffic
        base_rate = traffic_cfg.base_events_per_second[
            self.EVENT_TYPE.replace("-", "_")
        ]
        hour = datetime.now(timezone.utc).hour
        is_weekend = datetime.now(timezone.utc).weekday() >= 5

        multiplier = 1.0
        if hour in list(traffic_cfg.peak_hours):
            multiplier = traffic_cfg.peak_multiplier
        if is_weekend:
            multiplier *= traffic_cfg.weekend_dampener

        # Poisson process for realistic variability
        target = base_rate * self._batch_interval * multiplier
        actual = int(self._rng.poisson(max(target, 1)))
        return max(actual, 1)

    def _apply_edge_cases(self, records: List[Dict]) -> List[Dict]:

        for injector in self.edge_injectors:
            records = injector.inject(records, self._rng, self.metrics)
        return records

    def _add_to_dedup_buffer(self, event_id: str):
        self._recent_event_ids.append(event_id)
        if len(self._recent_event_ids) > self._dedup_buffer_size:
            self._recent_event_ids.pop(0)

    def _validate_records(self, records: List[Dict]) -> Tuple[List[Dict], List[Dict]]:

        if not self._validate_enabled or self.validator is None:
            return records, []
        valid, invalid = [], []
        for record in records:
            is_valid, errors = self.validator.validate(record, self.EVENT_TYPE)
            if is_valid:
                valid.append(record)
            else:
                record["_validation_errors"] = errors
                record["_is_valid"] = False
                self.metrics.records_failed_validation += 1
                if self.config.validation.strict_mode:
                    logger.warning(f"Dropping invalid record: {record['event_id']} errors={errors}")
                else:
                    # Tag and pass through for debugging/monitoring
                    valid.append(record)
                    invalid.append(record)
        return valid, invalid

    def _write_batch(self, records: List[Dict]) -> bool:

        if not records:
            return True

        # Rebuild output path in case hour/date changed mid-run
        output_dir = self._build_output_path()
        os.makedirs(output_dir, exist_ok=True)

        batch_id = str(uuid.uuid4())[:8]
        timestamp = int(time.time() * 1000)
        tmp_path = os.path.join(output_dir, f".{self.EVENT_TYPE}_{timestamp}_{batch_id}.tmp")
        final_path = os.path.join(output_dir, f"{self.EVENT_TYPE}_{timestamp}_{batch_id}.json")

        try:
            with open(tmp_path, "w") as f:
                for record in records:
                    f.write(json.dumps(record, default=str) + "\\n")
            os.rename(tmp_path, final_path)
            self.metrics.records_written += len(records)
            logger.debug(f"[{self.EVENT_TYPE}] Wrote {len(records)} records → {final_path}")
            return True
        except Exception as e:
            self.metrics.write_errors += 1
            logger.error(f"[{self.EVENT_TYPE}] Write failed: {e}")
            # Clean up tmp file if it exists
            if os.path.exists(tmp_path):
                try:
                    os.remove(tmp_path)
                except Exception:
                    pass
            return False

    def run_batch(self) -> int:
        batch_size = self._calculate_batch_size()
        records = []

        for _ in range(batch_size):
            record = self._generate_record()
            record["ingestion_timestamp"] = self.to_iso(datetime.now(timezone.utc))
            self._add_to_dedup_buffer(record["event_id"])
            records.append(record)
            self.metrics.records_generated += 1

        records = self._apply_edge_cases(records)
        valid_records, _ = self._validate_records(records)
        self._write_batch(valid_records)

        if self.config.platform.enable_metrics:
            self.metrics.log()

        return len(valid_records)
"""
path = "/mnt/c/Users/JatinJangid/.vscode/adtech-databricks/AdTech - Finance project/Adtech - generators/generator_src/core/base_generator.py"

with open(path, "w") as f:
    f.write(code.strip())
print(f"Written: {path}")

Written: /mnt/c/Users/JatinJangid/.vscode/adtech-databricks/AdTech - Finance project/Adtech - generators/generator_src/core/base_generator.py


In [4]:
code = """
import hashlib
import logging
import random
import uuid
from collections import defaultdict, deque
from dataclasses import dataclass, field
from datetime import datetime, timezone, timedelta
from typing import Any, Deque, Dict, List, Optional, Set
import numpy as np

logger = logging.getLogger(__name__)


@dataclass
class UserProfile:
    user_id: str
    is_bot: bool
    device_affinity: str           # primary device (but can switch)
    geo_country: str
    geo_city: str
    ip_address: str
    created_at: datetime
    session_count: int = 0
    last_seen: Optional[datetime] = None
    current_session_id: Optional[str] = None
    session_start: Optional[datetime] = None
    campaign_affinity: List[str] = field(default_factory=list)  # preferred campaigns


@dataclass
class CampaignState:
    campaign_id: str
    advertiser_id: str
    name: str
    category: str
    pricing_model: str             # CPC | CPM | CPA
    bid_price: float
    daily_budget: float
    total_budget: float
    spent_budget: float = 0.0
    status: str = "active"         # active | paused | ended
    created_at: datetime = field(default_factory=lambda: datetime.now(timezone.utc))
    last_modified: datetime = field(default_factory=lambda: datetime.now(timezone.utc))
    targeting_geos: List[str] = field(default_factory=list)
    targeting_devices: List[str] = field(default_factory=list)
    cdc_version: int = 1


@dataclass
class FunnelEvent:
    event_id: str
    event_type: str                # bid_request | impression | click | conversion
    campaign_id: str
    user_id: str
    session_id: str
    event_timestamp: datetime
    metadata: Dict = field(default_factory=dict)


class UserSimulator:

    DEVICE_TYPES = ["mobile", "desktop", "tablet"]
    GEO_CITY_MAP = {
        "US": ["New York", "Los Angeles", "Chicago", "Houston", "Phoenix"],
        "GB": ["London", "Manchester", "Birmingham", "Glasgow", "Leeds"],
        "DE": ["Berlin", "Munich", "Hamburg", "Frankfurt", "Cologne"],
        "FR": ["Paris", "Lyon", "Marseille", "Toulouse", "Nice"],
        "CA": ["Toronto", "Vancouver", "Montreal", "Calgary", "Ottawa"],
        "AU": ["Sydney", "Melbourne", "Brisbane", "Perth", "Adelaide"],
        "IN": ["Mumbai", "Delhi", "Bangalore", "Chennai", "Hyderabad"],
        "OTHER": ["Singapore", "Dubai", "Tokyo", "Seoul", "Amsterdam"],
    }

    def __init__(self, config, rng: np.random.Generator):
        self.config = config
        self._rng = rng
        self._users: Dict[str, UserProfile] = {}
        self._bot_ips: List[str] = []
        self._clean_ips: List[str] = []
        self._fraud_ips: List[str] = []

        self._initialize_ip_pools()
        self._initialize_user_pool()

    def _random_ip(self, reserved: bool = False) -> str:
        while True:
            octets = [
                int(self._rng.integers(1, 255)),
                int(self._rng.integers(0, 255)),
                int(self._rng.integers(0, 255)),
                int(self._rng.integers(1, 254)),
            ]
            # Skip private/loopback ranges
            if octets[0] in (10, 127) or (octets[0] == 172 and 16 <= octets[1] <= 31):
                continue
            if octets[0] == 192 and octets[1] == 168:
                continue
            return ".".join(map(str, octets))

    def _initialize_ip_pools(self):
        cfg = self.config.fraud
        # Clean user IPs
        pool_size = cfg.ip_pool_size
        fraud_pool_size = cfg.fraud_ip_pool_size
        self._clean_ips = [self._random_ip() for _ in range(pool_size - fraud_pool_size)]
        # Fraud IPs — small pool, high reuse (click farm signature)
        self._fraud_ips = [self._random_ip() for _ in range(fraud_pool_size)]
        logger.info(f"IP pools: {len(self._clean_ips)} clean, {len(self._fraud_ips)} fraud")

    def _initialize_user_pool(self):
        cfg = self.config.users
        pool_size = cfg.total_pool_size
        bot_count = int(pool_size * cfg.bot_user_ratio)
        geo_dist = self.config.users.geo_distribution

        geo_countries = list(geo_dist.to_dict().keys())
        geo_weights = list(geo_dist.to_dict().values())

        device_dist = cfg.device_distribution
        device_types = list(device_dist.to_dict().keys())
        device_weights = list(device_dist.to_dict().values())

        for i in range(pool_size):
            user_id = f"u_{str(uuid.UUID(int=int(hashlib.md5(str(i).encode()).hexdigest(), 16)))}"
            is_bot = i < bot_count

            country = self._rng.choice(geo_countries, p=geo_weights)
            city = self._rng.choice(self.GEO_CITY_MAP.get(country, ["Unknown"]))
            device = self._rng.choice(device_types, p=device_weights)
            ip = (
                self._rng.choice(self._fraud_ips)
                if is_bot
                else self._rng.choice(self._clean_ips)
            )

            self._users[user_id] = UserProfile(
                user_id=user_id,
                is_bot=is_bot,
                device_affinity=device,
                geo_country=country,
                geo_city=city,
                ip_address=ip,
                created_at=datetime.now(timezone.utc),
            )

        logger.info(f"User pool initialized: {pool_size} users ({bot_count} bots)")

    def get_or_create_user(self) -> UserProfile:

        cfg = self.config.users
        all_ids = list(self._users.keys())

        roll = self._rng.random()
        if roll < cfg.new_user_injection_rate:
            # Inject a truly new user mid-stream
            new_user = self._create_new_user()
            self._users[new_user.user_id] = new_user
            return new_user
        elif roll < cfg.new_user_injection_rate + (1 - cfg.returning_user_ratio):
            # Pick from recently created users (not returning)
            recent_ids = all_ids[-1000:]
            return self._users[str(self._rng.choice(recent_ids))]
        else:
            # Returning user — skewed toward more active users (Zipf)
            # Heavy users appear more often
            weights = np.ones(len(all_ids))
            # Assign higher weight to users with more sessions
            for idx, uid in enumerate(all_ids[:100]):  # top 100 are "heavy"
                weights[idx] = 5.0
            weights /= weights.sum()
            chosen_id = str(self._rng.choice(all_ids, p=weights))
            return self._users[chosen_id]

    def _create_new_user(self) -> UserProfile:
        cfg = self.config.users
        geo_dist = cfg.geo_distribution
        geo_countries = list(geo_dist.to_dict().keys())
        geo_weights = list(geo_dist.to_dict().values())
        device_dist = cfg.device_distribution
        device_types = list(device_dist.to_dict().keys())
        device_weights = list(device_dist.to_dict().values())

        user_id = f"u_{str(uuid.uuid4())[:12]}"
        country = str(self._rng.choice(geo_countries, p=geo_weights))
        city = str(self._rng.choice(self.GEO_CITY_MAP.get(country, ["Unknown"])))
        device = str(self._rng.choice(device_types, p=device_weights))
        ip = str(self._rng.choice(self._clean_ips))

        return UserProfile(
            user_id=user_id,
            is_bot=False,
            device_affinity=device,
            geo_country=country,
            geo_city=city,
            ip_address=ip,
            created_at=datetime.now(timezone.utc),
        )

    def get_or_create_session(self, user: UserProfile) -> str:
        timeout_minutes = self.config.users.session_timeout_minutes
        now = datetime.now(timezone.utc)

        if user.is_bot:
            timeout_minutes = 2  # bots cycle sessions aggressively

        session_expired = (
            user.session_start is None
            or (now - user.session_start) > timedelta(minutes=timeout_minutes)
        )

        if session_expired:
            user.current_session_id = f"s_{str(uuid.uuid4())[:16]}"
            user.session_start = now
            user.session_count += 1

        user.last_seen = now
        return user.current_session_id

    def get_device_for_event(self, user: UserProfile) -> str:
        if self._rng.random() < 0.10 and not user.is_bot:
            others = [d for d in self.DEVICE_TYPES if d != user.device_affinity]
            return str(self._rng.choice(others))
        return user.device_affinity

    def all_user_ids(self) -> List[str]:
        return list(self._users.keys())

    def get_user(self, user_id: str) -> Optional[UserProfile]:
        return self._users.get(user_id)


class CampaignStateManager:

    def __init__(self, config, rng: np.random.Generator):
        self.config = config
        self._rng = rng
        self._campaigns: Dict[str, CampaignState] = {}
        self._cdc_log: Deque[Dict] = deque(maxlen=10000)

        self._initialize_campaigns()

    def _generate_bid_price(self, pricing_model: str) -> float:
        pm_cfg = self.config.campaigns.pricing_models[pricing_model]
        dist = pm_cfg.bid_distribution
        if dist == "lognormal":
            price = float(self._rng.lognormal(
                mean=pm_cfg.bid_mu,
                sigma=pm_cfg.bid_sigma
            ))
            return round(max(pm_cfg.bid_floor, min(price, pm_cfg.bid_ceiling)), 4)
        return float(pm_cfg.bid_floor)

    def _initialize_campaigns(self):
        cfg = self.config.campaigns
        pm_list = list(cfg.pricing_models.to_dict().keys())
        pm_weights = [cfg.pricing_models[m].weight for m in pm_list]
        categories = list(cfg.categories)

        # Pareto budget distribution: top campaigns get most budget
        count = cfg.count
        pareto_alpha = cfg.pareto_alpha
        budgets = self._rng.pareto(pareto_alpha, count) + 1
        budgets = (budgets / budgets.sum()) * (count * 5000)  # total pool = N*5000

        for i in range(count):
            campaign_id = f"camp_{str(uuid.UUID(int=int(hashlib.md5(str(i*999).encode()).hexdigest(), 16)))[:8]}"
            advertiser_id = f"adv_{str(i // 5).zfill(4)}"  # ~10 campaigns per advertiser
            pricing_model = str(self._rng.choice(pm_list, p=pm_weights))
            status_roll = self._rng.random()
            lc = cfg.lifecycle
            if status_roll < lc.active_probability:
                status = "active"
            elif status_roll < lc.active_probability + lc.paused_probability:
                status = "paused"
            else:
                status = "ended"

            daily_budget = float(budgets[i])
            self._campaigns[campaign_id] = CampaignState(
                campaign_id=campaign_id,
                advertiser_id=advertiser_id,
                name=f"Campaign_{i}_{str(self._rng.choice(categories))}",
                category=str(self._rng.choice(categories)),
                pricing_model=pricing_model,
                bid_price=self._generate_bid_price(pricing_model),
                daily_budget=daily_budget,
                total_budget=daily_budget * float(self._rng.integers(7, 90)),
                status=status,
                targeting_geos=list(self._rng.choice(
                    list(self.config.users.geo_distribution.to_dict().keys()),
                    size=int(self._rng.integers(1, 5)),
                    replace=False
                )),
                targeting_devices=list(self._rng.choice(
                    ["mobile", "desktop", "tablet"],
                    size=int(self._rng.integers(1, 4)),
                    replace=False
                )),
            )

        logger.info(f"Campaign pool initialized: {count} campaigns")

    def get_active_campaign(self) -> CampaignState:
        active = [c for c in self._campaigns.values() if c.status == "active"]
        if not active:
            # Edge case: all campaigns paused — return any (for testing downstream)
            active = list(self._campaigns.values())

        # Weight by remaining budget (richer campaigns get more impressions)
        budgets = np.array([max(c.daily_budget - c.spent_budget, 0.01) for c in active])
        weights = budgets / budgets.sum()
        idx = int(self._rng.choice(len(active), p=weights))
        return active[idx]

    def get_campaign(self, campaign_id: str) -> Optional[CampaignState]:
        return self._campaigns.get(campaign_id)

    def debit_spend(self, campaign_id: str, amount: float):
        if campaign_id in self._campaigns:
            self._campaigns[campaign_id].spent_budget += amount

    def generate_cdc_event(self) -> Optional[Dict]:

        cfg = self.config.event_rates.campaign_cdc
        campaigns = list(self._campaigns.values())
        campaign = self._campaigns[str(self._rng.choice(list(self._campaigns.keys())))]

        change_type_roll = self._rng.random()
        before_state = self._campaign_snapshot(campaign)

        if change_type_roll < cfg.budget_update_pct:
            change_type = "budget_update"
            # Increase or decrease budget
            factor = float(self._rng.uniform(0.5, 2.0))
            campaign.daily_budget = round(campaign.daily_budget * factor, 2)

        elif change_type_roll < cfg.budget_update_pct + cfg.pricing_change_pct:
            change_type = "pricing_model_change"
            pm_list = list(self.config.campaigns.pricing_models.to_dict().keys())
            new_model = str(self._rng.choice([m for m in pm_list if m != campaign.pricing_model]))
            campaign.pricing_model = new_model
            campaign.bid_price = self._generate_bid_price(new_model)

        elif change_type_roll < cfg.budget_update_pct + cfg.pricing_change_pct + cfg.pause_activate_pct:
            change_type = "status_change"
            campaign.status = "paused" if campaign.status == "active" else "active"

        else:
            change_type = "targeting_update"
            campaign.targeting_geos = list(self._rng.choice(
                list(self.config.users.geo_distribution.to_dict().keys()),
                size=int(self._rng.integers(1, 5)),
                replace=False
            ))

        campaign.last_modified = datetime.now(timezone.utc)
        campaign.cdc_version += 1

        after_state = self._campaign_snapshot(campaign)
        cdc_event = {
            "event_id": str(uuid.uuid4()),
            "event_type": "campaign_cdc",
            "event_timestamp": campaign.last_modified.strftime("%Y-%m-%dT%H:%M:%S.%f")[:-3] + "Z",
            "ingestion_timestamp": None,
            "campaign_id": campaign.campaign_id,
            "advertiser_id": campaign.advertiser_id,
            "change_type": change_type,
            "cdc_version": campaign.cdc_version,
            "before": before_state,
            "after": after_state,
            "is_fraud": False,
            "fraud_label": "clean",
        }
        self._cdc_log.append(cdc_event)
        return cdc_event

    def _campaign_snapshot(self, c: CampaignState) -> Dict:
        return {
            "pricing_model": c.pricing_model,
            "bid_price": c.bid_price,
            "daily_budget": c.daily_budget,
            "total_budget": c.total_budget,
            "status": c.status,
            "targeting_geos": c.targeting_geos,
            "targeting_devices": c.targeting_devices,
        }

    def all_campaign_ids(self) -> List[str]:
        return list(self._campaigns.keys())


class FunnelStateStore:

    MAX_BUFFER_PER_STAGE = 5000
    MAX_PENDING_CONVERSIONS = 10000

    def __init__(self):
        self._bid_requests: Deque[FunnelEvent] = deque(maxlen=self.MAX_BUFFER_PER_STAGE)
        self._impressions: Deque[FunnelEvent] = deque(maxlen=self.MAX_BUFFER_PER_STAGE)
        self._clicks: Deque[FunnelEvent] = deque(maxlen=self.MAX_BUFFER_PER_STAGE)
        # Pending conversions: click_id → list of pending (delayed) conversions
        self._pending_conversions: Dict[str, List[Dict]] = {}

    def add(self, stage: str, event: FunnelEvent):
        store = self._get_store(stage)
        if store is not None:
            store.append(event)

    def _get_store(self, stage: str) -> Optional[Deque]:
        return {
            "bid_request": self._bid_requests,
            "impression": self._impressions,
            "click": self._clicks,
        }.get(stage)

    def get_recent(self, stage: str, n: int = 1) -> List[FunnelEvent]:
        store = self._get_store(stage)
        if not store:
            return []
        items = list(store)
        if not items:
            return []
        # Return random recent items (not always the most recent — avoids artificial correlation)
        count = min(n, len(items))
        indices = np.random.choice(len(items), size=count, replace=False)
        return [items[i] for i in indices]

    def get_by_user(self, stage: str, user_id: str, n: int = 3) -> List[FunnelEvent]:
        store = self._get_store(stage)
        if not store:
            return []
        user_events = [e for e in store if e.user_id == user_id]
        return user_events[-n:] if user_events else []

    def register_pending_conversion(self, click_id: str, conversion_data: Dict):
        if len(self._pending_conversions) < self.MAX_PENDING_CONVERSIONS:
            if click_id not in self._pending_conversions:
                self._pending_conversions[click_id] = []
            self._pending_conversions[click_id].append(conversion_data)

    def pop_due_conversions(self, now: datetime) -> List[Dict]:
        due = []
        to_remove = []
        for click_id, conversions in self._pending_conversions.items():
            still_pending = []
            for conv in conversions:
                if datetime.fromisoformat(conv["_emit_after"]) <= now:
                    due.append(conv)
                else:
                    still_pending.append(conv)
            if still_pending:
                self._pending_conversions[click_id] = still_pending
            else:
                to_remove.append(click_id)
        for k in to_remove:
            del self._pending_conversions[k]
        return due

    def count(self, stage: str) -> int:
        store = self._get_store(stage)
        return len(store) if store else 0


class SharedGeneratorState:

    def __init__(self, config, rng: np.random.Generator):
        self.config = config
        self.users = UserSimulator(config, rng)
        self.campaigns = CampaignStateManager(config, rng)
        self.funnel = FunnelStateStore()
"""
path = "/mnt/c/Users/JatinJangid/.vscode/adtech-databricks/AdTech - Finance project/Adtech - generators/generator_src/simulation/shared_state.py"

with open(path, "w") as f:
    f.write(code.strip())
print(f"Written: {path}")

Written: /mnt/c/Users/JatinJangid/.vscode/adtech-databricks/AdTech - Finance project/Adtech - generators/generator_src/simulation/shared_state.py


In [5]:
code = """
import logging
import time
from dataclasses import dataclass, field
from datetime import datetime, timezone, timedelta
from typing import Dict, List, Optional, Tuple
import numpy as np

logger = logging.getLogger(__name__)


@dataclass
class BurstEvent:
    start_time: float          # monotonic clock
    duration_seconds: float
    multiplier: float
    cause: str
    active: bool = True


class BurstTrafficController:

    BURST_PROFILES = {
        "flash_sale": {
            "multiplier_range": (10, 50),
            "duration_range_sec": (60, 300),
            "shape": "flat",
            "probability_per_batch": 0.001,
        },
        "bot_farm": {
            "multiplier_range": (50, 200),
            "duration_range_sec": (10, 45),
            "shape": "cliff",  # sudden on, sudden off
            "probability_per_batch": 0.0005,
        },
        "viral_news": {
            "multiplier_range": (5, 15),
            "duration_range_sec": (300, 3600),
            "shape": "bell",  # gradual rise, peak, gradual fall
            "probability_per_batch": 0.002,
        },
        "ddos": {
            "multiplier_range": (200, 1000),
            "duration_range_sec": (15, 90),
            "shape": "cliff",
            "probability_per_batch": 0.0001,
        },
        "promoted_post": {
            "multiplier_range": (3, 8),
            "duration_range_sec": (1800, 7200),
            "shape": "bell",
            "probability_per_batch": 0.003,
        },
    }

    def __init__(self, config, rng: np.random.Generator):
        self.config = config
        self._rng = rng
        self._active_bursts: List[BurstEvent] = []
        self._burst_history: List[BurstEvent] = []

    def maybe_trigger_burst(self) -> Optional[BurstEvent]:
        for profile_name, profile in self.BURST_PROFILES.items():
            if self._rng.random() < profile["probability_per_batch"]:
                mult = float(self._rng.uniform(*profile["multiplier_range"]))
                duration = float(self._rng.uniform(*profile["duration_range_sec"]))
                burst = BurstEvent(
                    start_time=time.monotonic(),
                    duration_seconds=duration,
                    multiplier=mult,
                    cause=profile_name,
                )
                self._active_bursts.append(burst)
                self._burst_history.append(burst)
                logger.warning(
                    f"[BURST] {profile_name} triggered: {mult:.1f}x for {duration:.0f}s"
                )
                return burst
        return None

    def get_current_multiplier(self) -> Tuple[float, Optional[str]]:

        now = time.monotonic()
        effective_multiplier = 1.0
        active_cause = None

        expired = []
        for burst in self._active_bursts:
            elapsed = now - burst.start_time
            if elapsed >= burst.duration_seconds:
                burst.active = False
                expired.append(burst)
                continue

            progress = elapsed / burst.duration_seconds
            profile = self.BURST_PROFILES.get(burst.cause, {})
            shape = profile.get("shape", "flat")

            if shape == "flat":
                shape_multiplier = burst.multiplier
            elif shape == "cliff":
                # Sudden on, sudden off — no ramp
                shape_multiplier = burst.multiplier
            elif shape == "bell":
                # Gaussian bell: peaks at midpoint
                # 4*sigma covers the duration (2 sigma each side of center)
                center = 0.5
                sigma = 0.2
                bell = np.exp(-0.5 * ((progress - center) / sigma) ** 2)
                shape_multiplier = 1.0 + (burst.multiplier - 1.0) * bell
            else:
                shape_multiplier = burst.multiplier

            effective_multiplier = max(effective_multiplier, shape_multiplier)
            active_cause = burst.cause

        for b in expired:
            self._active_bursts.remove(b)

        return effective_multiplier, active_cause

    def inject_burst_metadata(self, record: Dict, burst_cause: Optional[str]) -> Dict:
        if burst_cause:
            record["_burst_event"] = burst_cause
            record["_during_burst"] = True
        return record


class HotPartitionSimulator:

    HOT_KEY_PROFILES = {
        "campaign_concentration": {
            "field": "campaign_id",
            "hot_pct": 0.80,           # 80% of events use hot_value
            "probability": 0.001,       # low probability per batch
            "duration_batches": 50,
        },
        "publisher_concentration": {
            "field": "publisher_domain",
            "hot_pct": 0.70,
            "probability": 0.002,
            "duration_batches": 100,
        },
        "geo_flood": {
            "field": "geo_country",
            "hot_pct": 0.85,
            "probability": 0.001,
            "duration_batches": 30,
        },
        "device_skew": {
            "field": "device_type",
            "hot_pct": 0.92,
            "probability": 0.003,
            "duration_batches": 20,
        },
    }

    def __init__(self, config, rng: np.random.Generator, shared_state):
        self.config = config
        self._rng = rng
        self._shared_state = shared_state
        self._active_hot_keys: Dict[str, Dict] = {}  # field → {value, remaining_batches}
        self._batch_count = 0

    def tick(self):
        self._batch_count += 1
        expired = []
        for field, state in self._active_hot_keys.items():
            state["remaining_batches"] -= 1
            if state["remaining_batches"] <= 0:
                expired.append(field)
                logger.info(f"[HOT_PARTITION] {field}={state['value']} hot key expired")

        for f in expired:
            del self._active_hot_keys[f]

        # Maybe activate new hot keys
        for profile_name, profile in self.HOT_KEY_PROFILES.items():
            field = profile["field"]
            if field not in self._active_hot_keys and self._rng.random() < profile["probability"]:
                hot_value = self._pick_hot_value(field)
                if hot_value:
                    self._active_hot_keys[field] = {
                        "value": hot_value,
                        "hot_pct": profile["hot_pct"],
                        "remaining_batches": profile["duration_batches"],
                        "profile": profile_name,
                    }
                    logger.warning(
                        f"[HOT_PARTITION] {field}={hot_value} is now HOT "
                        f"({profile['hot_pct']*100:.0f}% of traffic)"
                    )

    def _pick_hot_value(self, field: str) -> Optional[str]:
        if field == "campaign_id":
            ids = self._shared_state.campaigns.all_campaign_ids()
            return str(self._rng.choice(ids)) if ids else None
        elif field == "publisher_domain":
            from generators.bid_request_generator import PUBLISHER_DOMAINS
            return str(self._rng.choice(PUBLISHER_DOMAINS))
        elif field == "geo_country":
            countries = list(self.config.users.geo_distribution.to_dict().keys())
            return str(self._rng.choice(countries))
        elif field == "device_type":
            return "mobile"
        return None

    def apply_hot_key(self, record: Dict, rng: np.random.Generator) -> Dict:
        for field, state in self._active_hot_keys.items():
            if field in record and rng.random() < state["hot_pct"]:
                record[field] = state["value"]
                record["_hot_partition_field"] = field
                record["_hot_partition_value"] = state["value"]
        return record
"""

path = "/mnt/c/Users/JatinJangid/.vscode/adtech-databricks/AdTech - Finance project/Adtech - generators/generator_src/simulation/traffic_dynamics.py"

with open(path, "w") as f:
    f.write(code.strip())
print(f"Written: {path}")

Written: /mnt/c/Users/JatinJangid/.vscode/adtech-databricks/AdTech - Finance project/Adtech - generators/generator_src/simulation/traffic_dynamics.py


In [6]:
# simulation/replay_manager.py
"""
Replay Manager with idempotency guarantees.

Production replay scenarios:
1. FULL_REPLAY: Re-run entire dataset (e.g., after pipeline bug fix)
2. RANGE_REPLAY: Re-run specific time range (e.g., fix a single hour's data)
3. PARTIAL_REPLAY: Re-run specific event types (e.g., only re-process clicks)
4. INCREMENTAL_REPLAY: Re-run from a checkpoint offset

Each replayed record gets a replay_id and replay_run_id.
The replay_run_id lets downstream systems detect and deduplicate
entire replay runs without scanning all event_ids.

Idempotency guarantee: Given the same seed + config + replay params,
the EXACT same records are produced in the EXACT same order.
This is critical for pipeline regression testing.

Replay detection markers let the Silver layer:
- Skip updating already-correct records (idempotent MERGE)
- Track which records have been replayed (audit trail)
- Alert when a replay produces different results than expected
"""
import hashlib
import json
import logging
import os
import uuid
from dataclasses import dataclass, field
from datetime import datetime, timezone, timedelta
from typing import Any, Dict, List, Optional, Tuple
import numpy as np

logger = logging.getLogger(__name__)


@dataclass
class ReplaySpec:
    replay_run_id: str
    replay_type: str                    # full | range | partial | incremental
    seed: int
    event_types: List[str]              # which event types to replay
    start_time: Optional[datetime]      # for range replay
    end_time: Optional[datetime]        # for range replay
    checkpoint_offset: Optional[int]    # for incremental replay
    num_batches: int
    reason: str                         # human-readable reason for replay
    created_at: datetime = field(default_factory=lambda: datetime.now(timezone.utc))


class ReplayManager:
    """
    Manages replay lifecycle and injects replay metadata into records.
    Maintains a replay log for audit purposes.
    """

    REPLAY_LOG_PATH = "/dbfs/data/adtech/replay_log.jsonl"

    def __init__(self, config):
        self.config = config
        self._active_replay: Optional[ReplaySpec] = None
        self._replay_record_count = 0

    def start_replay(
        self,
        replay_type: str = "full",
        seed: Optional[int] = None,
        event_types: Optional[List[str]] = None,
        start_time: Optional[datetime] = None,
        end_time: Optional[datetime] = None,
        checkpoint_offset: Optional[int] = None,
        num_batches: int = 100,
        reason: str = "manual_replay",
    ) -> ReplaySpec:

        spec = ReplaySpec(
            replay_run_id=str(uuid.uuid4()),
            replay_type=replay_type,
            seed=seed or self.config.platform.seed,
            event_types=event_types or [
                "bid_requests", "impressions", "clicks", "conversions", "campaign_cdc"
            ],
            start_time=start_time,
            end_time=end_time,
            checkpoint_offset=checkpoint_offset,
            num_batches=num_batches,
            reason=reason,
        )
        self._active_replay = spec
        self._replay_record_count = 0
        self._log_replay_start(spec)
        logger.info(f"[REPLAY] Started: run_id={spec.replay_run_id} type={replay_type} reason={reason}")
        return spec

    def end_replay(self):
        if self._active_replay:
            self._log_replay_end(self._active_replay, self._replay_record_count)
            logger.info(
                f"[REPLAY] Completed: run_id={self._active_replay.replay_run_id} "
                f"records={self._replay_record_count}"
            )
            self._active_replay = None

    def annotate_record(self, record: Dict) -> Dict:
        """Inject replay metadata into a record during replay mode."""
        if not self._active_replay:
            record["is_replay"] = False
            record["replay_run_id"] = None
            record["replay_idempotency_key"] = None
            return record

        spec = self._active_replay
        self._replay_record_count += 1

        # Deterministic idempotency key: hash of (event_id + replay_run_id)
        # Downstream MERGE ON event_id AND NOT is_replay can skip already-processed records
        idempotency_key = hashlib.sha256(
            f"{record['event_id']}:{spec.replay_run_id}".encode()
        ).hexdigest()[:16]

        record["is_replay"] = True
        record["replay_run_id"] = spec.replay_run_id
        record["replay_type"] = spec.replay_type
        record["replay_reason"] = spec.reason
        record["replay_idempotency_key"] = idempotency_key
        record["replay_sequence_num"] = self._replay_record_count

        # For range replay: filter out records outside the time window
        if spec.start_time or spec.end_time:
            try:
                ts = datetime.fromisoformat(record["event_timestamp"].replace("Z", "+00:00"))
                if spec.start_time and ts < spec.start_time:
                    record["_replay_out_of_range"] = True
                if spec.end_time and ts > spec.end_time:
                    record["_replay_out_of_range"] = True
            except (ValueError, KeyError):
                pass

        return record

    def _log_replay_start(self, spec: ReplaySpec):
        os.makedirs(os.path.dirname(self.REPLAY_LOG_PATH), exist_ok=True)
        entry = {
            "log_type": "replay_start",
            "replay_run_id": spec.replay_run_id,
            "replay_type": spec.replay_type,
            "seed": spec.seed,
            "event_types": spec.event_types,
            "num_batches": spec.num_batches,
            "reason": spec.reason,
            "created_at": spec.created_at.isoformat(),
        }
        with open(self.REPLAY_LOG_PATH, "a") as f:
            f.write(json.dumps(entry) + "\n")

    def _log_replay_end(self, spec: ReplaySpec, record_count: int):
        entry = {
            "log_type": "replay_end",
            "replay_run_id": spec.replay_run_id,
            "total_records": record_count,
            "completed_at": datetime.now(timezone.utc).isoformat(),
        }
        try:
            with open(self.REPLAY_LOG_PATH, "a") as f:
                f.write(json.dumps(entry) + "\n")
        except Exception as e:
            logger.error(f"Failed to write replay end log: {e}")

In [7]:
code = """
import hashlib
import json
import logging
import os
import uuid
from dataclasses import dataclass, field
from datetime import datetime, timezone, timedelta
from typing import Any, Dict, List, Optional, Tuple
import numpy as np

logger = logging.getLogger(__name__)


@dataclass
class ReplaySpec:
    replay_run_id: str
    replay_type: str                    # full | range | partial | incremental
    seed: int
    event_types: List[str]              # which event types to replay
    start_time: Optional[datetime]      # for range replay
    end_time: Optional[datetime]        # for range replay
    checkpoint_offset: Optional[int]    # for incremental replay
    num_batches: int
    reason: str                         # human-readable reason for replay
    created_at: datetime = field(default_factory=lambda: datetime.now(timezone.utc))


class ReplayManager:

    REPLAY_LOG_PATH = "/dbfs/data/adtech/replay_log.jsonl"

    def __init__(self, config):
        self.config = config
        self._active_replay: Optional[ReplaySpec] = None
        self._replay_record_count = 0

    def start_replay(
        self,
        replay_type: str = "full",
        seed: Optional[int] = None,
        event_types: Optional[List[str]] = None,
        start_time: Optional[datetime] = None,
        end_time: Optional[datetime] = None,
        checkpoint_offset: Optional[int] = None,
        num_batches: int = 100,
        reason: str = "manual_replay",
    ) -> ReplaySpec:

        spec = ReplaySpec(
            replay_run_id=str(uuid.uuid4()),
            replay_type=replay_type,
            seed=seed or self.config.platform.seed,
            event_types=event_types or [
                "bid_requests", "impressions", "clicks", "conversions", "campaign_cdc"
            ],
            start_time=start_time,
            end_time=end_time,
            checkpoint_offset=checkpoint_offset,
            num_batches=num_batches,
            reason=reason,
        )
        self._active_replay = spec
        self._replay_record_count = 0
        self._log_replay_start(spec)
        logger.info(f"[REPLAY] Started: run_id={spec.replay_run_id} type={replay_type} reason={reason}")
        return spec

    def end_replay(self):
        if self._active_replay:
            self._log_replay_end(self._active_replay, self._replay_record_count)
            logger.info(
                f"[REPLAY] Completed: run_id={self._active_replay.replay_run_id} "
                f"records={self._replay_record_count}"
            )
            self._active_replay = None

    def annotate_record(self, record: Dict) -> Dict:
        if not self._active_replay:
            record["is_replay"] = False
            record["replay_run_id"] = None
            record["replay_idempotency_key"] = None
            return record

        spec = self._active_replay
        self._replay_record_count += 1

        # Deterministic idempotency key: hash of (event_id + replay_run_id)
        # Downstream MERGE ON event_id AND NOT is_replay can skip already-processed records
        idempotency_key = hashlib.sha256(
            f"{record['event_id']}:{spec.replay_run_id}".encode()
        ).hexdigest()[:16]

        record["is_replay"] = True
        record["replay_run_id"] = spec.replay_run_id
        record["replay_type"] = spec.replay_type
        record["replay_reason"] = spec.reason
        record["replay_idempotency_key"] = idempotency_key
        record["replay_sequence_num"] = self._replay_record_count

        # For range replay: filter out records outside the time window
        if spec.start_time or spec.end_time:
            try:
                ts = datetime.fromisoformat(record["event_timestamp"].replace("Z", "+00:00"))
                if spec.start_time and ts < spec.start_time:
                    record["_replay_out_of_range"] = True
                if spec.end_time and ts > spec.end_time:
                    record["_replay_out_of_range"] = True
            except (ValueError, KeyError):
                pass

        return record

    def _log_replay_start(self, spec: ReplaySpec):
        os.makedirs(os.path.dirname(self.REPLAY_LOG_PATH), exist_ok=True)
        entry = {
            "log_type": "replay_start",
            "replay_run_id": spec.replay_run_id,
            "replay_type": spec.replay_type,
            "seed": spec.seed,
            "event_types": spec.event_types,
            "num_batches": spec.num_batches,
            "reason": spec.reason,
            "created_at": spec.created_at.isoformat(),
        }
        with open(self.REPLAY_LOG_PATH, "a") as f:
            f.write(json.dumps(entry) + "\\n")

    def _log_replay_end(self, spec: ReplaySpec, record_count: int):
        entry = {
            "log_type": "replay_end",
            "replay_run_id": spec.replay_run_id,
            "total_records": record_count,
            "completed_at": datetime.now(timezone.utc).isoformat(),
        }
        try:
            with open(self.REPLAY_LOG_PATH, "a") as f:
                f.write(json.dumps(entry) + "\\n")
        except Exception as e:
            logger.error(f"Failed to write replay end log: {e}")

"""

path = "/mnt/c/Users/JatinJangid/.vscode/adtech-databricks/AdTech - Finance project/Adtech - generators/generator_src/simulation/replay_manager.py"

with open(path, "w") as f:
    f.write(code.strip())
print(f"Written: {path}")

Written: /mnt/c/Users/JatinJangid/.vscode/adtech-databricks/AdTech - Finance project/Adtech - generators/generator_src/simulation/replay_manager.py


In [8]:
code = """
import abc
import logging
import uuid
from copy import deepcopy
from datetime import datetime, timezone, timedelta
from typing import Any, Dict, List, Tuple
import numpy as np

logger = logging.getLogger(__name__)


class BaseEdgeCaseInjector(abc.ABC):

    def __init__(self, config):
        self.config = config

    @abc.abstractmethod
    def inject(
        self,
        records: List[Dict],
        rng: np.random.Generator,
        metrics,
    ) -> List[Dict]:
        pass


class FraudInjector(BaseEdgeCaseInjector):

    FRAUD_PROFILES = {
        "bot_traffic": {
            "timing_jitter_ms": 50,        # near-zero variance
            "click_interval_ms": 200,      # suspiciously fast
            "session_page_views": 50,      # unrealistically high
        },
        "click_farm": {
            "burst_window_sec": 60,
            "clicks_per_ip_per_window": 25,
        },
        "impression_stuffing": {
            "multiplier_min": 5,
            "multiplier_max": 10,
        },
        "conversion_fraud": {
            "revenue_inflation": 2.5,
        },
    }

    def __init__(self, config, event_type: str, fraud_ips: List[str]):
        super().__init__(config)
        self._event_type = event_type
        self._fraud_ips = fraud_ips
        self._fraud_pct = config.event_rates[event_type.replace("-", "_")].fraud_percentage

    def inject(self, records: List[Dict], rng: np.random.Generator, metrics) -> List[Dict]:
        result = []
        for record in records:
            if rng.random() < self._fraud_pct:
                record = self._apply_fraud_pattern(record, rng)
                metrics.fraud_injected += 1
            result.append(record)
        return result

    def _apply_fraud_pattern(self, record: Dict, rng: np.random.Generator) -> Dict:
        record = deepcopy(record)
        patterns = list(self.FRAUD_PROFILES.keys())

        # Weight distribution: bot and click farm most common
        weights = [0.35, 0.35, 0.20, 0.10]
        pattern = str(rng.choice(patterns, p=weights))

        record["is_fraud"] = True
        record["fraud_label"] = pattern
        record["fraud_signals"] = {}

        if pattern == "bot_traffic":
            record["ip_address"] = str(rng.choice(self._fraud_ips))
            record["user_agent"] = "python-requests/2.28.0"
            record["fraud_signals"]["timing_variance_ms"] = float(
                rng.uniform(10, self.FRAUD_PROFILES["bot_traffic"]["timing_jitter_ms"])
            )
            record["fraud_signals"]["clicks_per_second"] = float(rng.uniform(8, 25))

        elif pattern == "click_farm":
            record["ip_address"] = str(rng.choice(self._fraud_ips[:5]))  # very small pool
            burst_count = int(rng.integers(
                self.FRAUD_PROFILES["click_farm"]["clicks_per_ip_per_window"] - 5,
                self.FRAUD_PROFILES["click_farm"]["clicks_per_ip_per_window"] + 10
            ))
            record["fraud_signals"]["ip_click_burst_count"] = burst_count
            record["fraud_signals"]["burst_window_sec"] = (
                self.FRAUD_PROFILES["click_farm"]["burst_window_sec"]
            )

        elif pattern == "impression_stuffing":
            mult = int(rng.integers(
                self.FRAUD_PROFILES["impression_stuffing"]["multiplier_min"],
                self.FRAUD_PROFILES["impression_stuffing"]["multiplier_max"]
            ))
            record["fraud_signals"]["impression_stuffing_multiplier"] = mult
            # Modify cost to reflect inflated impressions
            if "cost_micros" in record:
                record["cost_micros"] = int(record["cost_micros"] * mult)

        elif pattern == "conversion_fraud":
            inflation = self.FRAUD_PROFILES["conversion_fraud"]["revenue_inflation"]
            if "revenue" in record:
                record["revenue"] = round(record["revenue"] * inflation, 4)
            record["fraud_signals"]["missing_click_chain"] = True

        return record


class LateDataInjector(BaseEdgeCaseInjector):

    def __init__(self, config, event_type: str):
        super().__init__(config)
        rate_cfg = config.event_rates[event_type.replace("-", "_")]
        self._late_pct = rate_cfg.late_event_pct
        # late_max_minutes may be hours for conversions
        self._late_max_minutes = getattr(rate_cfg, "late_max_minutes", None)
        self._late_max_hours = getattr(rate_cfg, "late_max_hours", None)
        self._late_max_total_minutes = (
            (self._late_max_hours * 60) if self._late_max_hours
            else (self._late_max_minutes or 60)
        )

    def inject(self, records: List[Dict], rng: np.random.Generator, metrics) -> List[Dict]:
        result = []
        timestamps_for_ooo = []

        for record in records:
            if rng.random() < self._late_pct:
                # Exponential distribution: median delay is small, tail is long
                # scale = mean of exponential. We want median around 10% of max.
                scale = self._late_max_total_minutes * 0.15
                delay_minutes = float(rng.exponential(scale))
                delay_minutes = min(delay_minutes, self._late_max_total_minutes)

                # Parse and shift event_timestamp backward
                try:
                    ts = datetime.fromisoformat(
                        record["event_timestamp"].replace("Z", "+00:00")
                    )
                    late_ts = ts - timedelta(minutes=delay_minutes)
                    record["event_timestamp"] = late_ts.strftime("%Y-%m-%dT%H:%M:%S.%f")[:-3] + "Z"
                    record["_late_by_minutes"] = round(delay_minutes, 2)
                    metrics.late_events_injected += 1
                except (KeyError, ValueError) as e:
                    logger.warning(f"Late injection failed for record: {e}")

            result.append(record)

        # Out-of-order injection: randomly swap timestamps in ~5% of batches
        if len(result) > 2 and rng.random() < 0.05:
            result = self._inject_out_of_order(result, rng)

        return result

    def _inject_out_of_order(self, records: List[Dict], rng: np.random.Generator) -> List[Dict]:
        records = list(records)
        idx_a = int(rng.integers(0, len(records)))
        idx_b = int(rng.integers(0, len(records)))
        if idx_a != idx_b:
            ts_a = records[idx_a].get("event_timestamp")
            ts_b = records[idx_b].get("event_timestamp")
            if ts_a and ts_b:
                records[idx_a]["event_timestamp"] = ts_b
                records[idx_b]["event_timestamp"] = ts_a
                records[idx_a]["_out_of_order"] = True
                records[idx_b]["_out_of_order"] = True
        return records


class DuplicateInjector(BaseEdgeCaseInjector):
    def __init__(self, config, event_type: str, event_id_buffer: List[str]):
        super().__init__(config)
        rate_cfg = config.event_rates[event_type.replace("-", "_")]
        self._dup_pct = rate_cfg.duplicate_pct
        self._event_id_buffer = event_id_buffer  # reference to generator's buffer

    def inject(self, records: List[Dict], rng: np.random.Generator, metrics) -> List[Dict]:
        result = list(records)
        additions = []

        for record in records:
            if rng.random() < self._dup_pct and self._event_id_buffer:
                # Re-emit a previous event_id
                dup_event_id = str(rng.choice(self._event_id_buffer))
                dup = deepcopy(record)
                dup["event_id"] = dup_event_id
                dup["ingestion_timestamp"] = datetime.now(timezone.utc).strftime(
                    "%Y-%m-%dT%H:%M:%S.%f"
                )[:-3] + "Z"
                dup["_is_duplicate"] = True

                # Near-duplicate: slight timestamp drift (simulates retry)
                if rng.random() < 0.3:
                    try:
                        ts = datetime.fromisoformat(
                            dup["event_timestamp"].replace("Z", "+00:00")
                        )
                        drift_ms = int(rng.integers(-500, 500))
                        ts = ts + timedelta(milliseconds=drift_ms)
                        dup["event_timestamp"] = ts.strftime("%Y-%m-%dT%H:%M:%S.%f")[:-3] + "Z"
                        dup["_timestamp_drift_ms"] = drift_ms
                    except Exception:
                        pass

                additions.append(dup)
                metrics.duplicates_injected += 1

        return result + additions


class FinancialEdgeCaseInjector(BaseEdgeCaseInjector):
    COST_FIELDS = ["cost_micros", "bid_price", "revenue", "clearing_price"]

    def __init__(self, config):
        super().__init__(config)
        fc = config.edge_cases.financial
        self._zero_cost_pct = fc.zero_cost_campaign_pct
        self._negative_adj_pct = fc.negative_adjustment_pct
        self._overbilling_pct = fc.overbilling_scenario_pct
        self._rounding_pct = fc.currency_rounding_pct
        self._overbilling_mult = fc.max_overbilling_multiplier

    def inject(self, records: List[Dict], rng: np.random.Generator, metrics) -> List[Dict]:
        result = []
        for record in records:
            has_cost_field = any(f in record for f in self.COST_FIELDS)
            if not has_cost_field:
                result.append(record)
                continue

            roll = rng.random()
            if roll < self._zero_cost_pct:
                record = self._apply_zero_cost(record)
            elif roll < self._zero_cost_pct + self._negative_adj_pct:
                record = self._apply_negative_adjustment(record, rng)
            elif roll < self._zero_cost_pct + self._negative_adj_pct + self._overbilling_pct:
                record = self._apply_overbilling(record, rng)
            elif roll < self._zero_cost_pct + self._negative_adj_pct + self._overbilling_pct + self._rounding_pct:
                record = self._apply_rounding_artifact(record)

            result.append(record)
        return result

    def _apply_zero_cost(self, record: Dict) -> Dict:
        record = deepcopy(record)
        for field in self.COST_FIELDS:
            if field in record:
                record[field] = 0
        record["_financial_edge_case"] = "zero_cost"
        return record

    def _apply_negative_adjustment(self, record: Dict, rng: np.random.Generator) -> Dict:
        record = deepcopy(record)
        for field in self.COST_FIELDS:
            if field in record and record[field] > 0:
                # Negative adjustment = fraction of original cost reversed
                adj_pct = float(rng.uniform(0.1, 1.0))
                record[field] = -round(record[field] * adj_pct, 4)
        record["_financial_edge_case"] = "negative_adjustment"
        record["adjustment_reason"] = str(rng.choice(["fraud_reversal", "chargeback", "pricing_error"]))
        return record

    def _apply_overbilling(self, record: Dict, rng: np.random.Generator) -> Dict:
        record = deepcopy(record)
        mult = float(rng.uniform(1.01, self._overbilling_mult))
        for field in self.COST_FIELDS:
            if field in record and record[field] > 0:
                record[field] = round(record[field] * mult, 4)
        record["_financial_edge_case"] = "overbilling"
        record["_overbilling_multiplier"] = round(mult, 4)
        return record

    def _apply_rounding_artifact(self, record: Dict) -> Dict:
        record = deepcopy(record)
        for field in self.COST_FIELDS:
            if field in record and isinstance(record[field], (int, float)) and record[field] > 0:
                # Introduce floating-point rounding artifact
                record[field] = record[field] + 1e-10
        record["_financial_edge_case"] = "rounding_artifact"
        return record
"""
path = "/mnt/c/Users/JatinJangid/.vscode/adtech-databricks/AdTech - Finance project/Adtech - generators/generator_src/edge_cases/injectors.py"

with open(path, "w") as f:
    f.write(code.strip())
print(f"Written: {path}")

Written: /mnt/c/Users/JatinJangid/.vscode/adtech-databricks/AdTech - Finance project/Adtech - generators/generator_src/edge_cases/injectors.py


In [9]:
code = """
import logging
from copy import deepcopy
from datetime import datetime, timezone, timedelta
from typing import Any, Dict, List, Tuple
import numpy as np

from edge_cases.injectors import BaseEdgeCaseInjector

logger = logging.getLogger(__name__)

# Common timezone offsets in seconds (to simulate non-UTC recording)
TIMEZONE_OFFSETS_SECONDS = {
    "US/Eastern": -5 * 3600,
    "US/Pacific": -8 * 3600,
    "Europe/London": 0,
    "Europe/Berlin": 1 * 3600,
    "Asia/Kolkata": 5 * 3600 + 1800,
    "Asia/Tokyo": 9 * 3600,
    "Australia/Sydney": 10 * 3600,
}


class TimeChaosInjector(BaseEdgeCaseInjector):

    def __init__(self, config, event_type: str):
        super().__init__(config)
        tc = config.time_chaos
        self._enabled = tc.enabled
        self._clock_skew_pct = tc.clock_skew_pct
        self._processing_divergence_pct = tc.processing_divergence_pct
        self._ntp_rollback_pct = tc.ntp_rollback_pct
        self._timezone_confusion_pct = tc.timezone_confusion_pct
        self._multi_source_pct = tc.multi_source_skew_pct

        # Persistent skew values — simulate a real drifting clock
        # The same server maintains its skew across batches (not random each time)
        self._server_skew_seconds: Dict[str, float] = {}
        self._tz_offsets = list(TIMEZONE_OFFSETS_SECONDS.values())

    def inject(self, records: List[Dict], rng: np.random.Generator, metrics) -> List[Dict]:
        if not self._enabled:
            return records

        result = []
        for record in records:
            record = self._apply_time_chaos(record, rng)
            result.append(record)

        # NTP rollback: batch-level operation (affects a contiguous sequence)
        if len(result) > 3 and rng.random() < self._ntp_rollback_pct:
            result = self._inject_ntp_rollback(result, rng)

        return result

    def _apply_time_chaos(self, record: Dict, rng: np.random.Generator) -> Dict:
        record = deepcopy(record)
        roll = rng.random()
        cumulative = 0.0

        # Ensure both event_time and processing_time are always present
        # This is foundational — downstream systems should ALWAYS have both
        if "processing_timestamp" not in record:
            record["processing_timestamp"] = record.get("ingestion_timestamp") or record.get("event_timestamp")
        if "event_time_source" not in record:
            record["event_time_source"] = "client"  # default: client-recorded

        cumulative += self._clock_skew_pct
        if roll < cumulative:
            return self._apply_clock_skew(record, rng)

        cumulative += self._processing_divergence_pct
        if roll < cumulative:
            return self._apply_processing_divergence(record, rng)

        cumulative += self._timezone_confusion_pct
        if roll < cumulative:
            return self._apply_timezone_confusion(record, rng)

        cumulative += self._multi_source_pct
        if roll < cumulative:
            return self._apply_multi_source_skew(record, rng)

        return record

    def _apply_clock_skew(self, record: Dict, rng: np.random.Generator) -> Dict:
        source_server = record.get("ad_server_id", record.get("campaign_id", "default"))[:8]

        if source_server not in self._server_skew_seconds:
            # First time we see this server: assign it a persistent skew
            # Bimodal: most servers are within ±30s, some are wildly off (minutes)
            if rng.random() < 0.8:
                skew = float(rng.uniform(-30, 30))
            else:
                skew = float(rng.choice([-300, -120, 120, 300, 600]))
            self._server_skew_seconds[source_server] = skew

        skew_seconds = self._server_skew_seconds[source_server]
        try:
            ts = datetime.fromisoformat(record["event_timestamp"].replace("Z", "+00:00"))
            skewed_ts = ts + timedelta(seconds=skew_seconds)
            record["event_timestamp"] = skewed_ts.strftime("%Y-%m-%dT%H:%M:%S.%f")[:-3] + "Z"
            record["_time_chaos_type"] = "clock_skew"
            record["_clock_skew_seconds"] = round(skew_seconds, 2)
            record["_source_server"] = source_server
        except (ValueError, KeyError):
            pass

        return record

    def _apply_processing_divergence(self, record: Dict, rng: np.random.Generator) -> Dict:

        try:
            event_ts = datetime.fromisoformat(record["event_timestamp"].replace("Z", "+00:00"))

            # Processing lag: lognormal (median ~2min, tail up to 2 hours)
            lag_seconds = float(rng.lognormal(mean=4.5, sigma=1.8))  # median ~90s
            lag_seconds = min(lag_seconds, 7200)  # cap at 2 hours

            processing_ts = event_ts + timedelta(seconds=lag_seconds)
            record["processing_timestamp"] = processing_ts.strftime("%Y-%m-%dT%H:%M:%S.%f")[:-3] + "Z"
            record["ingestion_timestamp"] = record["processing_timestamp"]
            record["_time_chaos_type"] = "processing_divergence"
            record["_processing_lag_seconds"] = round(lag_seconds, 2)
            # If pipeline uses ingestion_time for windowing, this event falls in WRONG window
            record["_event_window"] = event_ts.strftime("%Y-%m-%dT%H:00:00Z")
            record["_processing_window"] = processing_ts.strftime("%Y-%m-%dT%H:00:00Z")
            record["_window_mismatch"] = record["_event_window"] != record["_processing_window"]
        except (ValueError, KeyError):
            pass

        return record

    def _apply_timezone_confusion(self, record: Dict, rng: np.random.Generator) -> Dict:
        tz_offset_seconds = float(rng.choice(self._tz_offsets))
        try:
            ts = datetime.fromisoformat(record["event_timestamp"].replace("Z", "+00:00"))
            # The timestamp was recorded as local time but stored as if it were UTC
            # So to "un-confuse" it, we'd add the offset back; we store the confused version
            confused_ts = ts + timedelta(seconds=tz_offset_seconds)
            record["event_timestamp"] = confused_ts.strftime("%Y-%m-%dT%H:%M:%S.%f")[:-3] + "Z"
            record["_time_chaos_type"] = "timezone_confusion"
            record["_tz_offset_seconds"] = tz_offset_seconds
            record["event_time_source"] = "client_local_time"  # the bug marker
        except (ValueError, KeyError):
            pass

        return record

    def _apply_multi_source_skew(self, record: Dict, rng: np.random.Generator) -> Dict:
        server_side_offset_seconds = float(rng.uniform(1, 10))
        try:
            ts = datetime.fromisoformat(record["event_timestamp"].replace("Z", "+00:00"))
            server_ts = ts + timedelta(seconds=server_side_offset_seconds)
            record["server_recorded_timestamp"] = server_ts.strftime("%Y-%m-%dT%H:%M:%S.%f")[:-3] + "Z"
            record["_time_chaos_type"] = "multi_source_skew"
            record["_source_time_delta_seconds"] = round(server_side_offset_seconds, 3)
            record["event_time_source"] = "client"
        except (ValueError, KeyError):
            pass

        return record

    def _inject_ntp_rollback(self, records: List[Dict], rng: np.random.Generator) -> List[Dict]:

        rollback_seconds = float(rng.uniform(5, 60))
        start_idx = int(rng.integers(0, max(1, len(records) // 2)))
        affected_count = int(rng.integers(2, min(10, len(records) - start_idx)))

        for i in range(start_idx, start_idx + affected_count):
            if i >= len(records):
                break
            try:
                ts = datetime.fromisoformat(
                    records[i]["event_timestamp"].replace("Z", "+00:00")
                )
                # Earlier records in the rolled-back sequence have LATER timestamps
                # (because the clock jumped back partway through)
                offset = rollback_seconds * (1 - (i - start_idx) / affected_count)
                rolled_ts = ts - timedelta(seconds=offset)
                records[i]["event_timestamp"] = rolled_ts.strftime("%Y-%m-%dT%H:%M:%S.%f")[:-3] + "Z"
                records[i]["_time_chaos_type"] = "ntp_rollback"
                records[i]["_ntp_rollback_seconds"] = round(rollback_seconds, 2)
                records[i]["_monotonicity_violation"] = True
            except (ValueError, KeyError):
                pass

        logger.debug(f"NTP rollback injected: {affected_count} records, {rollback_seconds:.1f}s rollback")
        return records
"""

path = "/mnt/c/Users/JatinJangid/.vscode/adtech-databricks/AdTech - Finance project/Adtech - generators/generator_src/edge_cases/time_chaos_injector.py"

with open(path, "w") as f:
    f.write(code.strip())
print(f"Written: {path}")

Written: /mnt/c/Users/JatinJangid/.vscode/adtech-databricks/AdTech - Finance project/Adtech - generators/generator_src/edge_cases/time_chaos_injector.py


In [10]:
from edge_cases.injectors import BaseEdgeCaseInjector\ncode = """
import logging
from copy import deepcopy
from datetime import datetime, timezone
from typing import Any, Dict, List, Optional, Tuple
import numpy as np

logger = logging.getLogger(__name__)


# Schema evolution timeline: list of (version, change_type, change_spec)
# Applied cumulatively — V3 record has all changes from V1, V2, V3.
SCHEMA_EVOLUTION_TIMELINE = {
    "bid_requests": [
        (1, "baseline", {}),
        (2, "add_optional_field", {
            "field": "supply_chain_object",
            "type": "string",
            "nullable": True,
            "default": None,
            "description": "OpenRTB Supply Chain Object (ads.txt)"
        }),
        (3, "add_optional_field", {
            "field": "bid_floor_currency",
            "type": "string",
            "nullable": True,
            "default": "USD",
        }),
        (4, "type_widening", {
            "field": "bid_price_micros",
            "old_type": "int32",
            "new_type": "int64",
            "description": "Exceeding int32 max for high-value bids"
        }),
        (5, "add_nested_object", {
            "field": "user_segments",
            "schema": {
                "segment_ids": ["list", "string"],
                "model_scores": {"type": "object"},
            },
            "description": "DSP audience segment data"
        }),
        (6, "rename_field", {
            "old_name": "targeting_match_score",
            "new_name": "relevance_score",
            "keep_old": True,  # transition period: keep both
        }),
        (7, "deprecate_field", {
            "field": "viewability_score",
            "replacement": "measured_viewability",
            "emit_null": True,
        }),
        (8, "add_array_field", {
            "field": "deal_ids",
            "element_type": "string",
            "nullable": True,
        }),
    ],
    "clicks": [
        (1, "baseline", {}),
        (2, "add_optional_field", {
            "field": "click_type",
            "type": "string",
            "nullable": True,
            "default": "standard",
        }),
        (3, "add_optional_field", {
            "field": "interaction_type",
            "type": "string",
            "nullable": True,
        }),
        (4, "type_narrowing", {  # DANGEROUS
            "field": "cost",
            "old_type": "float64",
            "new_type": "float32",
            "description": "DANGEROUS: reduces precision, causes rounding in aggregations",
        }),
        (5, "add_nested_object", {
            "field": "viewability_data",
            "schema": {
                "in_view_pct": "float",
                "time_in_view_ms": "int",
            },
        }),
    ],
    "conversions": [
        (1, "baseline", {}),
        (2, "add_optional_field", {
            "field": "order_id",
            "type": "string",
            "nullable": True,
        }),
        (3, "add_optional_field", {
            "field": "product_ids",
            "type": "array",
            "element_type": "string",
            "nullable": True,
        }),
        (4, "add_optional_field", {
            "field": "consent_string",
            "type": "string",
            "nullable": True,
            "description": "GDPR TCF 2.0 consent string",
        }),
        (5, "rename_field", {
            "old_name": "revenue",
            "new_name": "gross_revenue",
            "keep_old": True,
        }),
    ],
}


class SchemaEvolutionController:

    def __init__(self, config, rng: np.random.Generator):
        self.config = config
        self._rng = rng
        se_cfg = config.schema_evolution
        self._evolution_rate = se_cfg.version_advance_probability
        self._current_versions: Dict[str, int] = {}
        self._version_history: Dict[str, List[Tuple]] = {}

        # Initialize all event types at V1
        for event_type in SCHEMA_EVOLUTION_TIMELINE:
            self._current_versions[event_type] = 1

        logger.info("Schema evolution controller initialized at V1 for all event types")

    def maybe_advance_version(self, event_type: str):
        if self._rng.random() < self._evolution_rate:
            timeline = SCHEMA_EVOLUTION_TIMELINE.get(event_type, [])
            max_version = len(timeline)
            current = self._current_versions.get(event_type, 1)
            if current < max_version:
                new_version = current + 1
                self._current_versions[event_type] = new_version
                change = timeline[new_version - 1]
                logger.warning(
                    f"[SCHEMA EVOLUTION] {event_type} advanced to V{new_version}: "
                    f"{change[1]} — {change[2]}"
                )

    def get_current_version(self, event_type: str) -> int:
        return self._current_versions.get(event_type, 1)


class SchemaEvolutionInjector(BaseEdgeCaseInjector):

    def __init__(self, config, event_type: str, evolution_controller: SchemaEvolutionController):
        super().__init__(config)
        self._event_type = event_type
        self._controller = evolution_controller
        self._timeline = SCHEMA_EVOLUTION_TIMELINE.get(event_type, [])

    def inject(self, records: List[Dict], rng: np.random.Generator, metrics) -> List[Dict]:
        # Advance version once per batch
        self._controller.maybe_advance_version(self._event_type)
        current_version = self._controller.get_current_version(self._event_type)

        result = []
        for record in records:
            # Some records in the batch are still at old schema version
            # (simulate rolling deployment: not all producers upgraded yet)
            record_version = self._sample_record_version(current_version, rng)
            record = self._apply_schema_version(record, record_version, rng)
            result.append(record)

        return result

    def _sample_record_version(self, current_version: int, rng: np.random.Generator) -> int:

        if current_version == 1:
            return 1
        # 70% at current version, 25% at current-1, 5% at older versions
        weights = [0.05] * (current_version - 2) + [0.25, 0.70] if current_version > 2 else [0.30, 0.70]
        versions = list(range(1, current_version + 1))
        # Ensure weights sum to 1 and match versions length
        if len(weights) < len(versions):
            weights = [1.0 / len(versions)] * len(versions)
        weights = weights[-len(versions):]
        total = sum(weights)
        weights = [w / total for w in weights]
        return int(rng.choice(versions, p=weights))

    def _apply_schema_version(self, record: Dict, version: int, rng: np.random.Generator) -> Dict:
        record = deepcopy(record)
        record["schema_version"] = version

        for ver, change_type, spec in self._timeline:
            if ver > version:
                break
            record = self._apply_change(record, change_type, spec, rng)

        return record

    def _apply_change(self, record: Dict, change_type: str,
                      spec: Dict, rng: np.random.Generator) -> Dict:

        if change_type == "baseline":
            return record

        elif change_type == "add_optional_field":
            field = spec["field"]
            if field not in record:
                # 90% chance the new field is populated, 10% null (optional)
                if rng.random() < 0.9:
                    default = spec.get("default")
                    if spec.get("type") == "array":
                        record[field] = []
                    else:
                        record[field] = default
                else:
                    record[field] = None

        elif change_type == "type_widening":
            field = spec["field"]
            if field in record and record[field] is not None:
                # Safe: widen the type representation
                record[field] = int(record[field]) if "int" in spec["new_type"] else float(record[field])
                record[f"_{field}_type_widened"] = True

        elif change_type == "type_narrowing":
            field = spec["field"]
            if field in record and record[field] is not None:
                # DANGEROUS: loses precision
                try:
                    import struct
                    # Simulate float32 precision loss
                    val = float(record[field])
                    packed = struct.pack('f', val)
                    narrowed = struct.unpack('f', packed)[0]
                    record[field] = narrowed
                    record[f"_{field}_precision_loss"] = abs(val - narrowed)
                except Exception:
                    pass

        elif change_type == "add_nested_object":
            field = spec["field"]
            if field not in record:
                nested = {}
                for sub_field, sub_type in spec.get("schema", {}).items():
                    if sub_type == "float":
                        nested[sub_field] = round(float(rng.random()), 4)
                    elif sub_type == "int":
                        nested[sub_field] = int(rng.integers(0, 10000))
                    elif isinstance(sub_type, list) and sub_type[0] == "list":
                        nested[sub_field] = [str(rng.integers(1000, 9999)) for _ in range(int(rng.integers(0, 5)))]
                    elif sub_type == "object" or isinstance(sub_type, dict):
                        nested[sub_field] = {"score": round(float(rng.random()), 4)}
                    else:
                        nested[sub_field] = None
                record[field] = nested

        elif change_type == "rename_field":
            old_name = spec["old_name"]
            new_name = spec["new_name"]
            if old_name in record:
                record[new_name] = record[old_name]
                if not spec.get("keep_old", True):
                    del record[old_name]
                record[f"_{old_name}_deprecated"] = True

        elif change_type == "deprecate_field":
            field = spec["field"]
            if field in record and spec.get("emit_null", True):
                replacement = spec.get("replacement")
                if replacement:
                    record[replacement] = record[field]  # migrate value to new name
                record[field] = None  # null out deprecated field

        elif change_type == "add_array_field":
            field = spec["field"]
            if field not in record:
                # Arrays sometimes empty, sometimes populated
                count = int(rng.integers(0, 4))
                record[field] = [str(rng.integers(10000, 99999)) for _ in range(count)] if count > 0 else []

        return record

"""

path = "/mnt/c/Users/JatinJangid/.vscode/adtech-databricks/AdTech - Finance project/Adtech - generators/generator_src/edge_cases/schema_evolution_injector.py"

with open(path, "w") as f:
    f.write(code.strip())
print(f"Written: {path}")

Written: /mnt/c/Users/JatinJangid/.vscode/adtech-databricks/AdTech - Finance project/Adtech - generators/generator_src/edge_cases/schema_evolution_injector.py


In [11]:
code = """
import logging
import random
import string
import uuid
from copy import deepcopy
from datetime import datetime, timezone, timedelta
from typing import Any, Dict, List, Optional
import numpy as np

from edge_cases.injectors import BaseEdgeCaseInjector

logger = logging.getLogger(__name__)


class DataQualityInjector(BaseEdgeCaseInjector):

    DQ_FAILURE_TYPES = {
        "null_injection": 0.30,
        "corrupt_record": 0.20,
        "invalid_id_format": 0.15,
        "type_confusion": 0.15,
        "truncated_record": 0.10,
        "encoding_corruption": 0.05,
        "oversized_field": 0.03,
        "referential_integrity_violation": 0.02,
    }

    # Fields safe to null (non-critical path)
    NULLABLE_FIELDS = [
        "geo_city", "user_agent", "referrer_url", "landing_page_url",
        "ad_position", "viewability", "render_time_ms", "supply_chain_object",
        "bid_floor_currency", "user_segments", "deal_ids",
    ]

    # Fields that SHOULD NOT be null (tests downstream null-handling)
    CRITICAL_NULLABLE_FIELDS = [
        "campaign_id", "user_id", "device_type", "geo_country",
    ]

    def __init__(self, config, event_type: str):
        super().__init__(config)
        dq_cfg = config.data_quality
        self._injection_pct = dq_cfg.injection_pct
        self._critical_null_pct = dq_cfg.critical_null_pct
        self._failure_types = list(self.DQ_FAILURE_TYPES.keys())
        self._failure_weights = list(self.DQ_FAILURE_TYPES.values())

    def inject(self, records: List[Dict], rng: np.random.Generator, metrics) -> List[Dict]:
        result = []
        for record in records:
            if rng.random() < self._injection_pct:
                failure_type = str(rng.choice(self._failure_types, p=self._failure_weights))
                record = self._apply_dq_failure(record, failure_type, rng)
                record["_dq_failure_type"] = failure_type
            result.append(record)
        return result

    def _apply_dq_failure(self, record: Dict, failure_type: str,
                          rng: np.random.Generator) -> Dict:
        record = deepcopy(record)

        if failure_type == "null_injection":
            return self._inject_nulls(record, rng)

        elif failure_type == "corrupt_record":
            return self._corrupt_record(record, rng)

        elif failure_type == "invalid_id_format":
            return self._corrupt_ids(record, rng)

        elif failure_type == "type_confusion":
            return self._inject_type_confusion(record, rng)

        elif failure_type == "truncated_record":
            return self._truncate_record(record, rng)

        elif failure_type == "encoding_corruption":
            return self._corrupt_encoding(record, rng)

        elif failure_type == "oversized_field":
            return self._inject_oversized_field(record, rng)

        elif failure_type == "referential_integrity_violation":
            return self._violate_referential_integrity(record, rng)

        return record

    def _inject_nulls(self, record: Dict, rng: np.random.Generator) -> Dict:

        if rng.random() < self._critical_null_pct:
            # Null a critical field
            field = str(rng.choice(self.CRITICAL_NULLABLE_FIELDS))
            record[field] = None
            record["_null_critical_field"] = field
        else:
            # Null a non-critical field
            nullable = [f for f in self.NULLABLE_FIELDS if f in record]
            if nullable:
                field = str(rng.choice(nullable))
                record[field] = None
        return record

    def _corrupt_record(self, record: Dict, rng: np.random.Generator) -> Dict:
        corruption = str(rng.choice([
            "future_timestamp", "negative_cost", "impossible_ctr",
            "negative_bid", "zero_campaign_id", "past_epoch_timestamp",
        ]))

        if corruption == "future_timestamp":
            future = datetime.now(timezone.utc) + timedelta(days=365)
            record["event_timestamp"] = future.strftime("%Y-%m-%dT%H:%M:%S.%f")[:-3] + "Z"

        elif corruption == "negative_cost":
            for field in ["cost", "cost_micros", "bid_price", "clearing_price"]:
                if field in record and record[field] is not None:
                    record[field] = -abs(record[field])

        elif corruption == "impossible_ctr":
            # CTR > 100% — impossible but sometimes appears due to dedup bugs
            record["_simulated_ctr"] = float(rng.uniform(1.5, 10.0))

        elif corruption == "negative_bid":
            if "bid_price" in record:
                record["bid_price"] = -float(rng.uniform(0.01, 5.0))

        elif corruption == "zero_campaign_id":
            record["campaign_id"] = "00000000-0000-0000-0000-000000000000"

        elif corruption == "past_epoch_timestamp":
            # Timestamp from 1970-01-01 (epoch zero — common bug in uninitialized vars)
            record["event_timestamp"] = "1970-01-01T00:00:00.000Z"

        record["_corruption_type"] = corruption
        return record

    def _corrupt_ids(self, record: Dict, rng: np.random.Generator) -> Dict:
        corruption = str(rng.choice([
            "truncated_uuid", "sql_injection", "xss_payload",
            "non_uuid_format", "empty_string", "numeric_string",
        ]))

        id_fields = [f for f in ["event_id", "campaign_id", "user_id", "session_id"]
                     if f in record]
        if not id_fields:
            return record

        target_field = str(rng.choice(id_fields))

        if corruption == "truncated_uuid":
            original = record[target_field]
            record[target_field] = original[:12] if original else "truncated"

        elif corruption == "sql_injection":
            record[target_field] = "'; DROP TABLE events; --"

        elif corruption == "xss_payload":
            record[target_field] = "<script>alert('xss')</script>"

        elif corruption == "non_uuid_format":
            record[target_field] = "INVALID_ID_" + "".join(
                random.choices(string.ascii_uppercase, k=8)
            )

        elif corruption == "empty_string":
            record[target_field] = ""

        elif corruption == "numeric_string":
            record[target_field] = str(int(rng.integers(1, 999999)))

        record["_id_corruption"] = corruption
        record["_corrupted_field"] = target_field
        return record

    def _inject_type_confusion(self, record: Dict, rng: np.random.Generator) -> Dict:

        confusion = str(rng.choice(["numeric_as_string", "bool_as_int", "float_as_int"]))

        if confusion == "numeric_as_string":
            for field in ["cost", "bid_price", "clearing_price", "revenue"]:
                if field in record and record[field] is not None:
                    record[field] = str(record[field])  # "1.234" instead of 1.234
                    break

        elif confusion == "bool_as_int":
            for field in ["is_fraud", "_is_duplicate", "missing_impression"]:
                if field in record and isinstance(record[field], bool):
                    record[field] = 1 if record[field] else 0
                    break

        elif confusion == "float_as_int":
            for field in ["viewability", "targeting_match_score"]:
                if field in record and record[field] is not None:
                    record[field] = int(record[field] * 100)  # 0.75 → 75
                    break

        record["_type_confusion"] = confusion
        return record

    def _truncate_record(self, record: Dict, rng: np.random.Generator) -> Dict:

        all_fields = list(record.keys())
        keep_count = max(3, int(rng.integers(3, len(all_fields) // 2 + 1)))
        # Always keep event_id and event_timestamp (most parsers expect them)
        required = {"event_id", "event_timestamp", "event_type"}
        optional_fields = [f for f in all_fields if f not in required]
        keep_optional = list(rng.choice(
            optional_fields,
            size=max(0, keep_count - len(required)),
            replace=False
        ))
        keep_fields = required | set(keep_optional)
        truncated = {k: v for k, v in record.items() if k in keep_fields}
        truncated["_truncated_record"] = True
        truncated["_original_field_count"] = len(all_fields)
        truncated["_remaining_field_count"] = len(keep_fields)
        return truncated

    def _corrupt_encoding(self, record: Dict, rng: np.random.Generator) -> Dict:

        corruption_chars = [
            "\\x00",           # null byte — breaks many string operations
            "\ufffd",          # unicode replacement character
            "\xe2\x80\x8b",   # zero-width space (invisible, causes length mismatches)
            "🚀💥",            # emoji (4-byte UTF8, breaks naive string length assumptions)
            "\\r\\n",            # Windows line endings in a JSON string field\n        ]

        string_fields = [
            k for k, v in record.items()
            if isinstance(v, str) and k not in ("event_id", "event_timestamp", "event_type")
        ]
        if string_fields:
            target = str(rng.choice(string_fields))
            corruption = str(rng.choice(corruption_chars))
            record[target] = record[target] + corruption if record[target] else corruption
            record["_encoding_corruption_field"] = target

        return record

    def _inject_oversized_field(self, record: Dict, rng: np.random.Generator) -> Dict:
        size_kb = int(rng.choice([1, 10, 100, 1000]))  # 1KB to 1MB
        oversized_value = "X" * (size_kb * 1024)
        record["_oversized_payload"] = oversized_value[:100] + "...[TRUNCATED]"
        record["_oversized_field_kb"] = size_kb
        # Don't actually write the full payload — just mark that it happened
        return record

    def _violate_referential_integrity(self, record: Dict,
                                       rng: np.random.Generator) -> Dict:

        violation = str(rng.choice(["nonexistent_campaign", "nonexistent_user", "nonexistent_session"]))

        if violation == "nonexistent_campaign":
            record["campaign_id"] = f"GHOST_CAMP_{str(uuid.uuid4())[:8]}"
            record["_ref_integrity_violation"] = "campaign_id_not_in_dim_campaigns"

        elif violation == "nonexistent_user":
            record["user_id"] = f"GHOST_USER_{str(uuid.uuid4())[:8]}"
            record["_ref_integrity_violation"] = "user_id_not_in_dim_users"

        elif violation == "nonexistent_session":
            record["session_id"] = f"GHOST_SESSION_{str(uuid.uuid4())[:8]}"
            record["_ref_integrity_violation"] = "session_id_orphaned"

        return record
"""

path = "/mnt/c/Users/JatinJangid/.vscode/adtech-databricks/AdTech - Finance project/Adtech - generators/generator_src/edge_cases/data_quality_injector.py"

with open(path, "w") as f:
    f.write(code.strip())
print(f"Written: {path}")

Written: /mnt/c/Users/JatinJangid/.vscode/adtech-databricks/AdTech - Finance project/Adtech - generators/generator_src/edge_cases/data_quality_injector.py


In [12]:
code = """
import logging
from datetime import datetime, timezone, timedelta
from typing import Any, Dict, List, Optional, Tuple

logger = logging.getLogger(__name__)

try:
    import jsonschema
    JSONSCHEMA_AVAILABLE = True
except ImportError:
    JSONSCHEMA_AVAILABLE = False
    logger.warning("jsonschema not available; structural validation disabled")


BUSINESS_RULES = {
    "bid_request": [
        lambda r: (r.get("bid_price", 0) >= 0, "bid_price must be non-negative"),
        lambda r: (r.get("floor_price_cpm", 0) >= 0, "floor_price_cpm must be non-negative"),
        lambda r: (r.get("event_id") is not None, "event_id is required"),
    ],
    "impression": [
        lambda r: (r.get("clearing_price", 0) >= 0, "clearing_price must be non-negative"),
        lambda r: (r.get("viewability", 1) <= 1.0, "viewability must be <= 1.0"),
    ],
    "click": [
        lambda r: (r.get("cost", 0) >= 0 or r.get("_financial_edge_case") == "negative_adjustment",
                   "cost must be non-negative unless negative_adjustment"),
        lambda r: (r.get("time_to_click_seconds", 1) >= 0, "time_to_click must be non-negative"),
    ],
    "conversion": [
        lambda r: (r.get("revenue") is not None, "revenue is required"),
        lambda r: (r.get("attribution_window_hours", 1) > 0, "attribution_window must be positive"),
    ],
    "campaign_cdc": [
        lambda r: (r.get("change_type") in
                   ["budget_update", "pricing_model_change", "status_change",
                    "targeting_update", "heartbeat"],
                   "invalid change_type"),
    ],
}

SCHEMAS: Dict[str, Dict] = {
    "bid_request": {
        "type": "object",
        "required": ["event_id", "event_timestamp", "campaign_id", "user_id",
                     "device_type", "bid_price", "pricing_model"],
        "properties": {
            "event_id": {"type": "string", "minLength": 1},
            "bid_price": {"type": "number", "minimum": 0},
            "device_type": {"enum": ["mobile", "desktop", "tablet"]},
            "pricing_model": {"enum": ["CPC", "CPM", "CPA"]},
        },
        "additionalProperties": True,
    },
    "impression": {
        "type": "object",
        "required": ["event_id", "event_timestamp", "campaign_id", "user_id", "clearing_price"],
        "properties": {
            "clearing_price": {"type": "number", "minimum": 0},
            "viewability": {"type": "number", "minimum": 0, "maximum": 1},
        },
        "additionalProperties": True,
    },
    "click": {
        "type": "object",
        "required": ["event_id", "event_timestamp", "campaign_id", "user_id"],
        "additionalProperties": True,
    },
    "conversion": {
        "type": "object",
        "required": ["event_id", "event_timestamp", "campaign_id", "user_id", "revenue"],
        "properties": {
            "revenue": {"type": "number"},
        },
        "additionalProperties": True,
    },
    "campaign_cdc": {
        "type": "object",
        "required": ["event_id", "event_timestamp", "campaign_id", "change_type"],
        "additionalProperties": True,
    },
}


class SchemaValidator:

    def __init__(self, config, max_record_age_hours: int = 72):
        self._config = config
        self._max_age = timedelta(hours=max_record_age_hours)

    def validate(self, record: Dict, event_type: str) -> Tuple[bool, List[str]]:
        errors = []

        # 1. JSON Schema structural validation
        if JSONSCHEMA_AVAILABLE and event_type in SCHEMAS:
            try:
                jsonschema.validate(record, SCHEMAS[event_type])
            except jsonschema.ValidationError as e:
                errors.append(f"schema: {e.message}")
            except jsonschema.SchemaError as e:
                logger.error(f"Invalid schema definition for {event_type}: {e}")

        # 2. Business rule validation
        for rule_fn in BUSINESS_RULES.get(event_type, []):
            try:
                passed, message = rule_fn(record)
                if not passed:
                    errors.append(f"business_rule: {message}")
            except Exception as e:
                errors.append(f"rule_error: {e}")

        # 3. Temporal sanity check (very old events are suspicious unless flagged)
        if "event_timestamp" in record and record.get("_late_by_minutes") is None:
            try:
                ts = datetime.fromisoformat(
                    record["event_timestamp"].replace("Z", "+00:00")
                )
                age = datetime.now(timezone.utc) - ts
                if age > self._max_age:
                    errors.append(f"temporal: event_timestamp too old ({age.total_seconds()/3600:.1f}h)")
            except ValueError:
                errors.append("temporal: invalid event_timestamp format")

        return len(errors) == 0, errors
"""

path = "/mnt/c/Users/JatinJangid/.vscode/adtech-databricks/AdTech - Finance project/Adtech - generators/generator_src/validation/schema_validator.py"

with open(path, "w") as f:
    f.write(code.strip())
print(f"Written: {path}")

Written: /mnt/c/Users/JatinJangid/.vscode/adtech-databricks/AdTech - Finance project/Adtech - generators/generator_src/validation/schema_validator.py


In [13]:
code = """
import uuid
from datetime import datetime, timezone
from typing import Any, Dict, List
import numpy as np

from core.base_generator import BaseEventGenerator
from simulation.shared_state import SharedGeneratorState, FunnelEvent
from edge_cases.injectors import (
    FraudInjector, LateDataInjector, DuplicateInjector, FinancialEdgeCaseInjector
)


PUBLISHER_DOMAINS = [
    "news.example.com", "sports.example.net", "tech.example.org",
    "entertainment.example.com", "finance.example.net", "health.example.org",
    "travel.example.com", "shopping.example.net", "gaming.example.io",
]

AD_SIZES = ["300x250", "728x90", "320x50", "160x600", "970x250", "300x600"]

AD_POSITIONS = ["above_fold", "below_fold", "sidebar", "footer", "interstitial"]


class BidRequestGenerator(BaseEventGenerator):

    EVENT_TYPE = "bid_requests"

    def __init__(self, config, shared_state: SharedGeneratorState, validator=None):
        # Build edge case injectors in pipeline order
        fraud_ips = shared_state.users._fraud_ips
        edge_injectors = [
            FraudInjector(config, self.EVENT_TYPE, fraud_ips),
            LateDataInjector(config, self.EVENT_TYPE),
            DuplicateInjector(config, self.EVENT_TYPE, self._recent_event_ids_ref()),
            FinancialEdgeCaseInjector(config),
        ]
        super().__init__(config, shared_state, edge_injectors, validator)

    def _recent_event_ids_ref(self):
        # Lazy reference — populated after super().__init__
        return self._recent_event_ids

    def _generate_record(self) -> Dict[str, Any]:
        campaign = self.shared_state.campaigns.get_active_campaign()
        user = self.shared_state.users.get_or_create_user()
        session_id = self.shared_state.users.get_or_create_session(user)
        device = self.shared_state.users.get_device_for_event(user)
        now = self.current_event_time()

        event_id = self.generate_event_id()
        publisher_domain = str(self._rng.choice(PUBLISHER_DOMAINS))
        ad_size = str(self._rng.choice(AD_SIZES))
        ad_position = str(self._rng.choice(AD_POSITIONS))

        # Floor price: lognormal, lower bound at $0.01
        floor_price_cpm = max(
            0.01,
            float(self._rng.lognormal(mean=0.5, sigma=0.7))
        )

        # Bid price from campaign config
        pm = campaign.pricing_model
        pm_cfg = self.config.campaigns.pricing_models[pm]
        bid_price = max(
            pm_cfg.bid_floor,
            min(float(self._rng.lognormal(pm_cfg.bid_mu, pm_cfg.bid_sigma)), pm_cfg.bid_ceiling)
        )

        # Store in funnel state for downstream generators
        funnel_event = FunnelEvent(
            event_id=event_id,
            event_type="bid_request",
            campaign_id=campaign.campaign_id,
            user_id=user.user_id,
            session_id=session_id,
            event_timestamp=now,
            metadata={"bid_price": bid_price, "floor_price_cpm": floor_price_cpm},
        )
        self.shared_state.funnel.add("bid_request", funnel_event)

        return {
            "event_id": event_id,
            "event_type": "bid_request",
            "event_timestamp": self.to_iso(now),
            "ingestion_timestamp": None,  # filled by base class
            "campaign_id": campaign.campaign_id,
            "advertiser_id": campaign.advertiser_id,
            "user_id": user.user_id,
            "session_id": session_id,
            "device_type": device,
            "geo_country": user.geo_country,
            "geo_city": user.geo_city,
            "ip_address": user.ip_address,
            "publisher_domain": publisher_domain,
            "ad_size": ad_size,
            "ad_position": ad_position,
            "floor_price_cpm": round(floor_price_cpm, 6),
            "bid_price": round(bid_price, 6),
            "bid_price_micros": int(bid_price * 1_000_000),
            "pricing_model": campaign.pricing_model,
            "targeting_match_score": round(float(self._rng.beta(2, 5)), 4),
            "user_agent": self._generate_user_agent(device, user.is_bot),
            "page_url": f"https://{publisher_domain}/article/{str(uuid.uuid4())[:8]}",
            "inventory_type": str(self._rng.choice(["display", "video", "native", "rich_media"],
                                                   p=[0.55, 0.25, 0.15, 0.05])),
            "viewability_score": round(float(self._rng.beta(3, 2)), 4),
            "is_fraud": False,
            "fraud_label": "clean",
            "fraud_signals": {},
            "_is_duplicate": False,
            "_late_by_minutes": None,
            "_out_of_order": False,
            "_financial_edge_case": None,
        }

    def _generate_user_agent(self, device: str, is_bot: bool) -> str:
        if is_bot:
            return "python-requests/2.28.0"
        agents = {
            "mobile": [
                "Mozilla/5.0 (iPhone; CPU iPhone OS 16_0 like Mac OS X) AppleWebKit/605.1.15",
                "Mozilla/5.0 (Linux; Android 13; Pixel 7) AppleWebKit/537.36",
                "Mozilla/5.0 (Linux; Android 12; Samsung SM-G991B) AppleWebKit/537.36",
            ],
            "desktop": [
                "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 Chrome/108.0",
                "Mozilla/5.0 (Macintosh; Intel Mac OS X 13_0) AppleWebKit/605.1.15",
                "Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.36 Firefox/108.0",
            ],
            "tablet": [
                "Mozilla/5.0 (iPad; CPU OS 16_0 like Mac OS X) AppleWebKit/605.1.15",
                "Mozilla/5.0 (Linux; Android 12; SM-T870) AppleWebKit/537.36",
            ],
        }
        pool = agents.get(device, agents["desktop"])
        return str(self._rng.choice(pool))

    def _get_schema(self) -> Dict:
        return {
            "type": "object",
            "required": ["event_id", "event_timestamp", "campaign_id", "user_id",
                         "device_type", "bid_price"],
            "properties": {
                "event_id": {"type": "string"},
                "event_timestamp": {"type": "string", "format": "date-time"},
                "bid_price": {"type": "number", "minimum": 0},
            }
        }
"""

path = "/mnt/c/Users/JatinJangid/.vscode/adtech-databricks/AdTech - Finance project/Adtech - generators/generator_src/generators/bid_request_generator.py"

with open(path, "w") as f:
    f.write(code.strip())
print(f"Written: {path}")

Written: /mnt/c/Users/JatinJangid/.vscode/adtech-databricks/AdTech - Finance project/Adtech - generators/generator_src/generators/bid_request_generator.py


In [14]:
code = """
import uuid
from datetime import datetime, timezone, timedelta
from typing import Any, Dict, List, Optional
import numpy as np

from core.base_generator import BaseEventGenerator
from simulation.shared_state import SharedGeneratorState, FunnelEvent
from edge_cases.injectors import (
    FraudInjector, LateDataInjector, DuplicateInjector, FinancialEdgeCaseInjector
)


class ImpressionGenerator(BaseEventGenerator):

    EVENT_TYPE = "impressions"

    def __init__(self, config, shared_state: SharedGeneratorState, validator=None):
        fraud_ips = shared_state.users._fraud_ips
        edge_injectors = [
            FraudInjector(config, self.EVENT_TYPE, fraud_ips),
            LateDataInjector(config, self.EVENT_TYPE),
            DuplicateInjector(config, self.EVENT_TYPE, self._recent_event_ids_ref()),
            FinancialEdgeCaseInjector(config),
        ]
        super().__init__(config, shared_state, edge_injectors, validator)
        self._win_rate = config.event_rates.impressions.win_rate

    def _recent_event_ids_ref(self):
        return self._recent_event_ids

    def _generate_record(self) -> Dict[str, Any]:
        now = self.current_event_time()

        # Try to source from a real bid request (causal chain)
        bid_requests = self.shared_state.funnel.get_recent("bid_request", n=1)
        missing_bid_request = False

        if bid_requests and self._rng.random() < self._win_rate:
            source_bid = bid_requests[0]
            bid_request_id = source_bid.event_id
            campaign_id = source_bid.campaign_id
            user_id = source_bid.user_id
            session_id = source_bid.session_id
            # Impression happens shortly after bid (RTB latency: 50-200ms)
            impression_ts = source_bid.event_timestamp + timedelta(
                milliseconds=int(self._rng.integers(50, 200))
            )
            user = self.shared_state.users.get_user(user_id)
            device = self.shared_state.users.get_device_for_event(user) if user else "mobile"
            geo_country = user.geo_country if user else "US"
            geo_city = user.geo_city if user else "New York"
            ip_address = user.ip_address if user else "0.0.0.0"
            bid_price = source_bid.metadata.get("bid_price", 0.50)
            floor_price = source_bid.metadata.get("floor_price_cpm", 0.10)
        else:
            # Orphaned impression (no bid request reference)
            missing_bid_request = True
            bid_request_id = None
            campaign = self.shared_state.campaigns.get_active_campaign()
            user = self.shared_state.users.get_or_create_user()
            session_id = self.shared_state.users.get_or_create_session(user)
            campaign_id = campaign.campaign_id
            impression_ts = now
            device = self.shared_state.users.get_device_for_event(user)
            geo_country = user.geo_country
            geo_city = user.geo_city
            ip_address = user.ip_address
            bid_price = 0.50
            floor_price = 0.10
            user_id = user.user_id

        # Clearing price: second-price auction (between floor and bid)
        clearing_price = round(
            float(self._rng.uniform(floor_price / 1000, bid_price * 0.95)), 6
        )
        # CPM cost = clearing_price * 1000 / 1000 = clearing_price per impression
        cost_cpm = clearing_price
        # Debiting campaign budget
        self.shared_state.campaigns.debit_spend(campaign_id, cost_cpm / 1000)

        event_id = self.generate_event_id()
        funnel_event = FunnelEvent(
            event_id=event_id,
            event_type="impression",
            campaign_id=campaign_id,
            user_id=user_id,
            session_id=session_id,
            event_timestamp=impression_ts,
            metadata={"clearing_price": clearing_price, "bid_request_id": bid_request_id},
        )
        self.shared_state.funnel.add("impression", funnel_event)

        return {
            "event_id": event_id,
            "event_type": "impression",
            "event_timestamp": self.to_iso(impression_ts),
            "ingestion_timestamp": None,
            "bid_request_id": bid_request_id,
            "campaign_id": campaign_id,
            "user_id": user_id,
            "session_id": session_id,
            "device_type": device,
            "geo_country": geo_country,
            "geo_city": geo_city,
            "ip_address": ip_address,
            "clearing_price": clearing_price,
            "clearing_price_micros": int(clearing_price * 1_000_000),
            "cost_cpm": round(cost_cpm, 6),
            "bid_price": round(bid_price, 6),
            "viewability": round(float(self._rng.beta(3, 2)), 4),
            "render_time_ms": int(self._rng.lognormal(mean=4.5, sigma=0.8)),
            "ad_server_id": f"as_{str(uuid.uuid4())[:8]}",
            "missing_bid_request": missing_bid_request,
            "auction_type": str(self._rng.choice(["first_price", "second_price"], p=[0.4, 0.6])),
            "is_fraud": False,
            "fraud_label": "clean",
            "fraud_signals": {},
            "_is_duplicate": False,
            "_late_by_minutes": None,
            "_financial_edge_case": None,
        }

    def _get_schema(self) -> Dict:
        return {
            "type": "object",
            "required": ["event_id", "event_timestamp", "campaign_id", "user_id", "clearing_price"],
        }
"""

path = "/mnt/c/Users/JatinJangid/.vscode/adtech-databricks/AdTech - Finance project/Adtech - generators/generator_src/generators/impression_generator.py"

with open(path, "w") as f:
    f.write(code.strip())
print(f"Written: {path}")

Written: /mnt/c/Users/JatinJangid/.vscode/adtech-databricks/AdTech - Finance project/Adtech - generators/generator_src/generators/impression_generator.py


In [15]:
code = """
import uuid
from datetime import datetime, timezone, timedelta
from typing import Any, Dict, Optional
import numpy as np

from core.base_generator import BaseEventGenerator
from simulation.shared_state import SharedGeneratorState, FunnelEvent
from edge_cases.injectors import (
    FraudInjector, LateDataInjector, DuplicateInjector, FinancialEdgeCaseInjector
)


class ClickGenerator(BaseEventGenerator):

    EVENT_TYPE = "clicks"

    def __init__(self, config, shared_state: SharedGeneratorState, validator=None):
        fraud_ips = shared_state.users._fraud_ips
        edge_injectors = [
            FraudInjector(config, self.EVENT_TYPE, fraud_ips),
            LateDataInjector(config, self.EVENT_TYPE),
            DuplicateInjector(config, self.EVENT_TYPE, self._recent_event_ids_ref()),
            FinancialEdgeCaseInjector(config),
        ]
        super().__init__(config, shared_state, edge_injectors, validator)
        rate_cfg = config.event_rates.clicks
        self._missing_impression_pct = rate_cfg.missing_impression_pct
        self._ctr_base = rate_cfg.ctr_base

    def _recent_event_ids_ref(self):
        return self._recent_event_ids

    def _generate_record(self) -> Dict[str, Any]:
        now = self.current_event_time()

        # Try to source from a real impression
        impressions = self.shared_state.funnel.get_recent("impression", n=1)
        missing_impression = False

        if impressions and self._rng.random() > self._missing_impression_pct:
            source_imp = impressions[0]
            impression_id = source_imp.event_id
            campaign_id = source_imp.campaign_id
            user_id = source_imp.user_id
            session_id = source_imp.session_id
            # Click happens seconds to minutes after impression
            click_ts = source_imp.event_timestamp + timedelta(
                seconds=int(self._rng.lognormal(mean=2.5, sigma=1.2))
            )
            user = self.shared_state.users.get_user(user_id)
            device = self.shared_state.users.get_device_for_event(user) if user else "mobile"
            geo_country = user.geo_country if user else "US"
            geo_city = user.geo_city if user else "New York"
            ip_address = user.ip_address if user else "0.0.0.0"
            clearing_price = source_imp.metadata.get("clearing_price", 0.50)
            # CPC cost = bid price (first price) or clearing price
            campaign = self.shared_state.campaigns.get_campaign(campaign_id)
            cost = campaign.bid_price if campaign else clearing_price
        else:
            # Orphaned click — no impression reference
            missing_impression = True
            impression_id = None
            campaign = self.shared_state.campaigns.get_active_campaign()
            user = self.shared_state.users.get_or_create_user()
            session_id = self.shared_state.users.get_or_create_session(user)
            campaign_id = campaign.campaign_id
            user_id = user.user_id
            click_ts = now
            device = self.shared_state.users.get_device_for_event(user)
            geo_country = user.geo_country
            geo_city = user.geo_city
            ip_address = user.ip_address
            cost = campaign.bid_price

        # Debit CPC cost
        if campaign and campaign.pricing_model == "CPC":
            self.shared_state.campaigns.debit_spend(campaign_id, cost)

        event_id = self.generate_event_id()
        funnel_event = FunnelEvent(
            event_id=event_id,
            event_type="click",
            campaign_id=campaign_id,
            user_id=user_id,
            session_id=session_id,
            event_timestamp=click_ts,
            metadata={
                "cost": cost,
                "impression_id": impression_id,
                "geo_country": geo_country,
            },
        )
        self.shared_state.funnel.add("click", funnel_event)

        # Click coordinates (realistic on-ad click position)
        click_x = int(self._rng.integers(0, 300))
        click_y = int(self._rng.integers(0, 250))

        # Bot clicks have suspiciously uniform coordinates
        if self._rng.random() < self.config.event_rates.clicks.fraud_percentage * 0.5:
            click_x, click_y = 150, 125  # dead center every time

        return {
            "event_id": event_id,
            "event_type": "click",
            "event_timestamp": self.to_iso(click_ts),
            "ingestion_timestamp": None,
            "impression_id": impression_id,
            "campaign_id": campaign_id,
            "user_id": user_id,
            "session_id": session_id,
            "device_type": device,
            "geo_country": geo_country,
            "geo_city": geo_city,
            "ip_address": ip_address,
            "cost": round(cost, 6),
            "cost_micros": int(cost * 1_000_000),
            "click_x": click_x,
            "click_y": click_y,
            "time_to_click_seconds": round(
                float(self._rng.lognormal(mean=2.5, sigma=1.2)), 2
            ),
            "landing_page_url": f"https://advertiser-{campaign_id[:8]}.example.com/landing",
            "referrer_url": f"https://pub.example.com/page/{str(uuid.uuid4())[:8]}",
            "missing_impression": missing_impression,
            "is_fraud": False,
            "fraud_label": "clean",
            "fraud_signals": {},
            "_is_duplicate": False,
            "_late_by_minutes": None,
            "_financial_edge_case": None,
        }

    def _get_schema(self) -> Dict:
        return {
            "type": "object",
            "required": ["event_id", "event_timestamp", "campaign_id", "user_id", "cost"],
        }
"""

path = "/mnt/c/Users/JatinJangid/.vscode/adtech-databricks/AdTech - Finance project/Adtech - generators/generator_src/generators/click_generator.py"

with open(path, "w") as f:
    f.write(code.strip())
print(f"Written: {path}")

Written: /mnt/c/Users/JatinJangid/.vscode/adtech-databricks/AdTech - Finance project/Adtech - generators/generator_src/generators/click_generator.py


In [16]:
code = """
import uuid
from datetime import datetime, timezone, timedelta
from typing import Any, Dict, List, Optional
import numpy as np

from core.base_generator import BaseEventGenerator
from simulation.shared_state import SharedGeneratorState, FunnelEvent
from edge_cases.injectors import (
    FraudInjector, LateDataInjector, DuplicateInjector, FinancialEdgeCaseInjector
)


CONVERSION_TYPES = [
    ("purchase", 0.35),
    ("signup", 0.25),
    ("lead_form", 0.20),
    ("app_install", 0.10),
    ("add_to_cart", 0.07),
    ("checkout_start", 0.03),
]


class ConversionGenerator(BaseEventGenerator):

    EVENT_TYPE = "conversions"

    def __init__(self, config, shared_state: SharedGeneratorState, validator=None):
        fraud_ips = shared_state.users._fraud_ips
        edge_injectors = [
            FraudInjector(config, self.EVENT_TYPE, fraud_ips),
            LateDataInjector(config, self.EVENT_TYPE),
            DuplicateInjector(config, self.EVENT_TYPE, self._recent_event_ids_ref()),
            FinancialEdgeCaseInjector(config),
        ]
        super().__init__(config, shared_state, edge_injectors, validator)
        rate_cfg = config.event_rates.conversions
        self._missing_click_pct = rate_cfg.missing_click_pct
        self._delayed_conv_pct = rate_cfg.delayed_conversion_pct
        self._delayed_max_hours = rate_cfg.delayed_conversion_max_hours
        self._cvr_base = rate_cfg.cvr_base

        self._conv_types = [t[0] for t in CONVERSION_TYPES]
        self._conv_weights = [t[1] for t in CONVERSION_TYPES]

    def _recent_event_ids_ref(self):
        return self._recent_event_ids

    def run_batch(self) -> int:
        # First, emit any pending delayed conversions whose window has passed
        now = datetime.now(timezone.utc)
        pending = self.shared_state.funnel.pop_due_conversions(now)
        if pending:
            for conv in pending:
                conv.pop("_emit_after", None)
                conv["ingestion_timestamp"] = self.to_iso(now)
            self._write_batch(pending)
            self.metrics.records_written += len(pending)

        # Then run normal batch
        return super().run_batch()

    def _generate_record(self) -> Dict[str, Any]:
        now = self.current_event_time()
        rate_cfg = self.config.event_rates.conversions

        # Source from a real click
        clicks = self.shared_state.funnel.get_recent("click", n=1)
        missing_click = False

        if clicks and self._rng.random() > self._missing_click_pct:
            source_click = clicks[0]
            click_id = source_click.event_id
            campaign_id = source_click.campaign_id
            user_id = source_click.user_id
            session_id = source_click.session_id
            user = self.shared_state.users.get_user(user_id)
            geo_country = user.geo_country if user else "US"
            geo_city = user.geo_city if user else "New York"
            ip_address = user.ip_address if user else "0.0.0.0"
            device = self.shared_state.users.get_device_for_event(user) if user else "mobile"

            # Cross-device check: 15% chance user converts on different device
            if self._rng.random() < 0.15 and user:
                devices = ["mobile", "desktop", "tablet"]
                other_devices = [d for d in devices if d != device]
                device = str(self._rng.choice(other_devices))
                session_id = f"s_{str(uuid.uuid4())[:16]}"  # new session for cross-device

            # Multi-touch: find all clicks for this user (attribution touch points)
            all_user_clicks = self.shared_state.funnel.get_by_user("click", user_id, n=5)
            touch_points = [
                {"click_id": c.event_id, "campaign_id": c.campaign_id,
                 "timestamp": self.to_iso(c.event_timestamp)}
                for c in all_user_clicks
            ]
        else:
            # Orphaned conversion — no click reference
            missing_click = True
            click_id = None
            campaign = self.shared_state.campaigns.get_active_campaign()
            user = self.shared_state.users.get_or_create_user()
            session_id = self.shared_state.users.get_or_create_session(user)
            campaign_id = campaign.campaign_id
            user_id = user.user_id
            device = self.shared_state.users.get_device_for_event(user)
            geo_country = user.geo_country
            geo_city = user.geo_city
            ip_address = user.ip_address
            touch_points = []

        campaign = self.shared_state.campaigns.get_campaign(campaign_id)
        conv_type = str(self._rng.choice(self._conv_types, p=self._conv_weights))

        # Revenue: lognormal (most conversions are small, occasional large purchase)
        revenue_base = {
            "purchase": (3.5, 1.2),
            "signup": (0.5, 0.3),
            "lead_form": (1.5, 0.8),
            "app_install": (2.0, 0.6),
            "add_to_cart": (3.0, 1.0),
            "checkout_start": (3.2, 1.0),
        }
        mu, sigma = revenue_base.get(conv_type, (2.0, 0.8))
        revenue = round(float(self._rng.lognormal(mean=mu, sigma=sigma)), 4)

        # CPA cost = bid price if CPA campaign
        cpa_cost = 0.0
        if campaign and campaign.pricing_model == "CPA":
            cpa_cost = campaign.bid_price
            self.shared_state.campaigns.debit_spend(campaign_id, cpa_cost)

        event_id = self.generate_event_id()
        conv_ts = now  # immediate (non-delayed)

        # Delayed conversion: register for future emission
        is_delayed = self._rng.random() < self._delayed_conv_pct
        if is_delayed:
            delay_hours = float(self._rng.exponential(scale=self._delayed_max_hours * 0.2))
            delay_hours = min(delay_hours, self._delayed_max_hours)
            emit_after = now + timedelta(hours=delay_hours)

            delayed_record = {
                "event_id": event_id,
                "event_type": "conversion",
                "event_timestamp": self.to_iso(now),
                "ingestion_timestamp": None,
                "click_id": click_id,
                "campaign_id": campaign_id,
                "user_id": user_id,
                "session_id": session_id,
                "device_type": device,
                "geo_country": geo_country,
                "geo_city": geo_city,
                "ip_address": ip_address,
                "conversion_type": conv_type,
                "revenue": revenue,
                "revenue_micros": int(revenue * 1_000_000),
                "cpa_cost": round(cpa_cost, 6),
                "attribution_model": str(self._rng.choice(
                    ["last_click", "first_click", "linear", "time_decay"],
                    p=[0.50, 0.20, 0.20, 0.10]
                )),
                "attribution_window_hours": 48,
                "touch_points": touch_points,
                "is_delayed_conversion": True,
                "delay_hours": round(delay_hours, 2),
                "missing_click": missing_click,
                "is_fraud": False,
                "fraud_label": "clean",
                "fraud_signals": {},
                "_is_duplicate": False,
                "_emit_after": emit_after.isoformat(),
                "_financial_edge_case": None,
            }
            if click_id:
                self.shared_state.funnel.register_pending_conversion(click_id, delayed_record)
            # Return a placeholder-free record to avoid double emission
            # We still need to return something — emit a different record
            return self._generate_record()  # recursive, bounded by low delayed_conv_pct

        return {
            "event_id": event_id,
            "event_type": "conversion",
            "event_timestamp": self.to_iso(conv_ts),
            "ingestion_timestamp": None,
            "click_id": click_id,
            "campaign_id": campaign_id,
            "user_id": user_id,
            "session_id": session_id,
            "device_type": device,
            "geo_country": geo_country,
            "geo_city": geo_city,
            "ip_address": ip_address,
            "conversion_type": conv_type,
            "revenue": revenue,
            "revenue_micros": int(revenue * 1_000_000),
            "cpa_cost": round(cpa_cost, 6),
            "attribution_model": str(self._rng.choice(
                ["last_click", "first_click", "linear", "time_decay"],
                p=[0.50, 0.20, 0.20, 0.10]
            )),
            "attribution_window_hours": 48,
            "touch_points": touch_points,
            "is_delayed_conversion": False,
            "delay_hours": 0.0,
            "missing_click": missing_click,
            "is_fraud": False,
            "fraud_label": "clean",
            "fraud_signals": {},
            "_is_duplicate": False,
            "_financial_edge_case": None,
        }

    def _get_schema(self) -> Dict:
        return {
            "type": "object",
            "required": ["event_id", "event_timestamp", "campaign_id", "user_id", "revenue"],
        }
"""
path = "/mnt/c/Users/JatinJangid/.vscode/adtech-databricks/AdTech - Finance project/Adtech - generators/generator_src/generators/conversion_generator.py"

with open(path, "w") as f:
    f.write(code.strip())
print(f"Written: {path}")

Written: /mnt/c/Users/JatinJangid/.vscode/adtech-databricks/AdTech - Finance project/Adtech - generators/generator_src/generators/conversion_generator.py


In [17]:
code = """
from datetime import datetime, timezone
from typing import Any, Dict
import numpy as np

from core.base_generator import BaseEventGenerator
from simulation.shared_state import SharedGeneratorState
from edge_cases.injectors import LateDataInjector


class CampaignCDCGenerator(BaseEventGenerator):

    EVENT_TYPE = "campaign_cdc"

    def __init__(self, config, shared_state: SharedGeneratorState, validator=None):
        edge_injectors = [
            LateDataInjector(config, self.EVENT_TYPE),
        ]
        super().__init__(config, shared_state, edge_injectors, validator)

    def _generate_record(self) -> Dict[str, Any]:
        cdc_event = self.shared_state.campaigns.generate_cdc_event()
        if cdc_event is None:
            # Fallback: generate a no-op heartbeat CDC event
            campaign = self.shared_state.campaigns.get_active_campaign()
            return {
                "event_id": self.generate_event_id(),
                "event_type": "campaign_cdc",
                "event_timestamp": self.to_iso(self.current_event_time()),
                "ingestion_timestamp": None,
                "campaign_id": campaign.campaign_id,
                "change_type": "heartbeat",
                "cdc_version": campaign.cdc_version,
                "before": {},
                "after": {},
                "is_fraud": False,
                "fraud_label": "clean",
            }
        return cdc_event

    def _get_schema(self) -> Dict:
        return {
            "type": "object",
            "required": ["event_id", "event_timestamp", "campaign_id", "change_type"],
        }
"""

path = "/mnt/c/Users/JatinJangid/.vscode/adtech-databricks/AdTech - Finance project/Adtech - generators/generator_src/generators/campaign_cdc_generator.py"

with open(path, "w") as f:
    f.write(code.strip())
print(f"Written: {path}")

Written: /mnt/c/Users/JatinJangid/.vscode/adtech-databricks/AdTech - Finance project/Adtech - generators/generator_src/generators/campaign_cdc_generator.py


In [18]:
code = """
import uuid
from datetime import datetime, timezone, timedelta
from typing import Any, Dict, List
import numpy as np

from core.base_generator import BaseEventGenerator
from simulation.shared_state import SharedGeneratorState
from edge_cases.injectors import LateDataInjector


REFUND_TYPES = {
    "full_refund": 0.30,
    "partial_refund": 0.35,
    "chargeback": 0.15,
    "fraud_reversal": 0.12,
    "goodwill_credit": 0.08,
}

CHARGEBACK_REASON_CODES = [
    "4853",   # Visa: Cardholder dispute (not as described)
    "4855",   # Visa: Goods/services not provided
    "4837",   # Mastercard: No cardholder authorization
    "4841",   # Mastercard: Cancelled recurring transaction
    "10.4",   # PayPal: Other fraud
]


class RefundGenerator(BaseEventGenerator):

    EVENT_TYPE = "refunds"

    def __init__(self, config, shared_state: SharedGeneratorState, validator=None):
        edge_injectors = [
            LateDataInjector(config, "conversions"),  # refunds arrive late too
        ]
        super().__init__(config, shared_state, edge_injectors, validator)
        self._refund_types = list(REFUND_TYPES.keys())
        self._refund_weights = list(REFUND_TYPES.values())

    def _generate_record(self) -> Dict[str, Any]:
        now = self.current_event_time()

        # Source from a real conversion (refunds reference conversions)
        conversions = self.shared_state.funnel.get_recent("click", n=1)
        # We use click buffer as proxy — in full system, would use conversion buffer
        has_source = bool(conversions)

        refund_type = str(self._rng.choice(self._refund_types, p=self._refund_weights))
        campaign = self.shared_state.campaigns.get_active_campaign()
        user = self.shared_state.users.get_or_create_user()

        if has_source:
            src = conversions[0]
            original_conversion_id = src.event_id
            campaign_id = src.campaign_id
            user_id = src.user_id
            original_amount = float(
                self._rng.lognormal(mean=3.5, sigma=1.2)
            )  # approximate original revenue
        else:
            original_conversion_id = f"CONV_{str(uuid.uuid4())[:12]}"
            campaign_id = campaign.campaign_id
            user_id = user.user_id
            original_amount = float(self._rng.lognormal(mean=3.5, sigma=1.2))

        # Refund delay: chargebacks arrive weeks later, fraud reversals are immediate
        delay_profile = {
            "full_refund": (1, 48),         # hours
            "partial_refund": (2, 72),
            "chargeback": (24 * 7, 24 * 90),  # 1-90 days
            "fraud_reversal": (0, 4),
            "goodwill_credit": (24, 168),     # 1-7 days
        }
        delay_min, delay_max = delay_profile[refund_type]
        delay_hours = float(self._rng.uniform(delay_min, delay_max))
        refund_ts = now - timedelta(hours=delay_hours)  # backdate the refund

        # Refund amount
        if refund_type == "full_refund":
            refund_amount = original_amount
            refund_pct = 1.0
        elif refund_type == "partial_refund":
            refund_pct = float(self._rng.uniform(0.1, 0.9))
            refund_amount = round(original_amount * refund_pct, 4)
        elif refund_type in ("chargeback", "fraud_reversal"):
            refund_amount = original_amount  # always full
            refund_pct = 1.0
        else:  # goodwill_credit
            refund_pct = float(self._rng.uniform(0.05, 0.30))
            refund_amount = round(original_amount * refund_pct, 4)

        # Attribution impact
        attribution_impact = {
            "full_refund": "full_reversal",
            "partial_refund": "partial_reversal",
            "chargeback": "full_reversal",
            "fraud_reversal": "full_reversal_and_flag",
            "goodwill_credit": "none",
        }[refund_type]

        record = {
            "event_id": self.generate_event_id(),
            "event_type": "refund",
            "event_timestamp": self.to_iso(refund_ts),
            "ingestion_timestamp": None,
            "original_conversion_id": original_conversion_id,
            "campaign_id": campaign_id,
            "user_id": user_id,
            "refund_type": refund_type,
            "original_amount": round(original_amount, 4),
            "refund_amount": round(refund_amount, 4),
            "refund_amount_micros": int(refund_amount * 1_000_000),
            "refund_pct": round(refund_pct, 4),
            "currency": "USD",
            "delay_hours": round(delay_hours, 2),
            "attribution_impact": attribution_impact,
            "requires_invoice_adjustment": refund_type in (
                "full_refund", "partial_refund", "chargeback"
            ),
            "requires_fraud_flag": refund_type in ("chargeback", "fraud_reversal"),
            "ledger_entry_type": "REVERSAL",
            "is_fraud": refund_type == "fraud_reversal",
            "fraud_label": "fraud_reversal" if refund_type == "fraud_reversal" else "clean",
            "fraud_signals": {},
            "_financial_edge_case": "refund",
        }

        # Chargeback-specific fields
        if refund_type == "chargeback":
            record["chargeback_reason_code"] = str(
                self._rng.choice(CHARGEBACK_REASON_CODES)
            )
            record["card_network"] = str(self._rng.choice(["visa", "mastercard", "amex"], p=[0.5, 0.35, 0.15]))
            record["dispute_status"] = str(self._rng.choice(
                ["open", "under_review", "resolved_merchant_win", "resolved_customer_win"],
                p=[0.3, 0.4, 0.2, 0.1]
            ))

        return record

    def _get_schema(self) -> Dict:
        return {
            "type": "object",
            "required": ["event_id", "event_timestamp", "original_conversion_id", "refund_amount"],
        }
"""

path = "/mnt/c/Users/JatinJangid/.vscode/adtech-databricks/AdTech - Finance project/Adtech - generators/generator_src/generators/refund_generator.py"

with open(path, "w") as f:
    f.write(code.strip())
print(f"Written: {path}")

Written: /mnt/c/Users/JatinJangid/.vscode/adtech-databricks/AdTech - Finance project/Adtech - generators/generator_src/generators/refund_generator.py


In [19]:
code = """
import uuid
import logging
from datetime import datetime, timezone, timedelta
from typing import Any, Dict, List, Optional
import numpy as np

from core.base_generator import BaseEventGenerator
from simulation.shared_state import SharedGeneratorState, FunnelEvent
from edge_cases.injectors import BaseEdgeCaseInjector, LateDataInjector

logger = logging.getLogger(__name__)

RECONCILIATION_FAILURE_TYPES = {
    "double_billing": 0.30,
    "phantom_billing": 0.15,
    "underbilling": 0.20,
    "ledger_mismatch": 0.15,
    "missing_ledger_entry": 0.10,
    "currency_conversion_drift": 0.05,
    "retroactive_price_change": 0.05,
}

SUPPORTED_CURRENCIES = {
    "USD": 1.0,
    "EUR": 0.92,
    "GBP": 0.79,
    "JPY": 148.5,
    "CAD": 1.35,
    "AUD": 1.53,
    "INR": 83.2,
}


class BillingRecordGenerator(BaseEventGenerator):

    EVENT_TYPE = "billing_records"

    def __init__(self, config, shared_state: SharedGeneratorState, validator=None):
        edge_injectors = [LateDataInjector(config, "conversions")]  # reuse conversion latency profile
        super().__init__(config, shared_state, edge_injectors, validator)
        self._recon_cfg = config.reconciliation
        self._failure_types = list(RECONCILIATION_FAILURE_TYPES.keys())
        self._failure_weights = list(RECONCILIATION_FAILURE_TYPES.values())
        self._currencies = list(SUPPORTED_CURRENCIES.keys())
        self._fx_rates = list(SUPPORTED_CURRENCIES.values())

        # FX rate snapshot at billing time (will drift vs ledger time)
        self._billing_fx_snapshot: Dict[str, float] = dict(SUPPORTED_CURRENCIES)

        # Buffer of billing records for generating matching ledger entries
        self._billing_buffer: List[Dict] = []

    def _generate_record(self) -> Dict[str, Any]:
        now = self.current_event_time()
        failure_pct = self._recon_cfg.failure_injection_pct
        roll = self._rng.random()

        if roll < failure_pct:
            return self._generate_failing_billing_record(now)
        else:
            return self._generate_clean_billing_record(now)

    def _generate_clean_billing_record(self, now: datetime) -> Dict:
        # Source from a real click or conversion
        source_events = (
            self.shared_state.funnel.get_recent("click", n=1) or
            self.shared_state.funnel.get_recent("impression", n=1)
        )
        campaign = self.shared_state.campaigns.get_active_campaign()

        if source_events:
            src = source_events[0]
            event_id = src.event_id
            campaign_id = src.campaign_id
            user_id = src.user_id
            event_type = src.event_type
            amount = src.metadata.get("cost", campaign.bid_price)
        else:
            event_id = self.generate_event_id()
            campaign_id = campaign.campaign_id
            user_id = f"u_{str(uuid.uuid4())[:12]}"
            event_type = "click"
            amount = campaign.bid_price

        billing_id = self.generate_event_id()
        currency = str(self._rng.choice(self._currencies[:3]))  # mostly USD/EUR/GBP
        fx_rate = self._billing_fx_snapshot.get(currency, 1.0)
        amount_local = round(amount * fx_rate, 6)

        record = {
            "billing_id": billing_id,
            "event_id": event_id,
            "event_type": event_type,
            "event_timestamp": self.to_iso(now - timedelta(seconds=int(self._rng.integers(1, 60)))),
            "billing_timestamp": self.to_iso(now),
            "ingestion_timestamp": None,
            "campaign_id": campaign_id,
            "advertiser_id": campaign.advertiser_id,
            "user_id": user_id,
            "amount_usd": round(amount, 6),
            "amount_local": amount_local,
            "currency": currency,
            "fx_rate": fx_rate,
            "pricing_model": campaign.pricing_model,
            "billing_status": "settled",
            "reconciliation_status": "pending",
            "is_reconciliation_failure": False,
            "failure_type": None,
            "failure_details": {},
        }
        self._billing_buffer.append(record)
        return record

    def _generate_failing_billing_record(self, now: datetime) -> Dict:
        failure_type = str(self._rng.choice(self._failure_types, p=self._failure_weights))
        campaign = self.shared_state.campaigns.get_active_campaign()
        base_amount = campaign.bid_price

        record = {
            "billing_id": self.generate_event_id(),
            "event_type": "click",
            "event_timestamp": self.to_iso(now - timedelta(seconds=int(self._rng.integers(1, 300)))),
            "billing_timestamp": self.to_iso(now),
            "ingestion_timestamp": None,
            "campaign_id": campaign.campaign_id,
            "advertiser_id": campaign.advertiser_id,
            "currency": "USD",
            "fx_rate": 1.0,
            "pricing_model": campaign.pricing_model,
            "billing_status": "settled",
            "is_reconciliation_failure": True,
            "failure_type": failure_type,
            "failure_details": {},
        }

        if failure_type == "double_billing":
            # Re-use an event_id from the buffer (same event billed twice)
            if self._billing_buffer:
                original = self._billing_buffer[int(self._rng.integers(0, len(self._billing_buffer)))]
                record["event_id"] = original["event_id"]
                record["amount_usd"] = original["amount_usd"]
                record["amount_local"] = original["amount_usd"]
                record["user_id"] = original.get("user_id", "unknown")
                record["failure_details"] = {
                    "original_billing_id": original["billing_id"],
                    "description": "Same event billed twice due to at-least-once delivery",
                }
            else:
                record["event_id"] = self.generate_event_id()
                record["amount_usd"] = round(base_amount, 6)
                record["amount_local"] = round(base_amount, 6)
                record["user_id"] = f"u_{str(uuid.uuid4())[:12]}"

        elif failure_type == "phantom_billing":
            # event_id that does NOT exist in the event stream
            record["event_id"] = f"PHANTOM_{str(uuid.uuid4())[:12]}"
            record["amount_usd"] = round(base_amount, 6)
            record["amount_local"] = round(base_amount, 6)
            record["user_id"] = f"u_{str(uuid.uuid4())[:12]}"
            record["failure_details"] = {
                "description": "Billing record references non-existent event_id",
                "likely_cause": "billing engine processed from wrong Kafka offset",
            }

        elif failure_type == "underbilling":
            # event_id exists, billing amount is less than it should be
            clicks = self.shared_state.funnel.get_recent("click", n=1)
            event_id = clicks[0].event_id if clicks else self.generate_event_id()
            correct_amount = campaign.bid_price
            billed_amount = round(correct_amount * float(self._rng.uniform(0.1, 0.8)), 6)
            record["event_id"] = event_id
            record["amount_usd"] = billed_amount
            record["amount_local"] = billed_amount
            record["user_id"] = clicks[0].user_id if clicks else f"u_{str(uuid.uuid4())[:12]}"
            record["failure_details"] = {
                "correct_amount": correct_amount,
                "billed_amount": billed_amount,
                "underbilling_delta": round(correct_amount - billed_amount, 6),
                "description": "Partial billing — billing engine crashed mid-batch",
            }

        elif failure_type == "ledger_mismatch":
            # Billing is correct but ledger will have different amount
            correct_amount = base_amount
            ledger_amount = round(correct_amount * float(self._rng.uniform(0.95, 1.05)), 6)
            record["event_id"] = self.generate_event_id()
            record["amount_usd"] = correct_amount
            record["amount_local"] = correct_amount
            record["user_id"] = f"u_{str(uuid.uuid4())[:12]}"
            record["failure_details"] = {
                "billing_amount": correct_amount,
                "expected_ledger_amount": ledger_amount,
                "delta": round(ledger_amount - correct_amount, 6),
                "description": "FX rate used in ledger differs from billing rate",
            }

        elif failure_type == "missing_ledger_entry":
            record["event_id"] = self.generate_event_id()
            record["amount_usd"] = round(base_amount, 6)
            record["amount_local"] = round(base_amount, 6)
            record["user_id"] = f"u_{str(uuid.uuid4())[:12]}"
            record["billing_status"] = "settled"
            record["failure_details"] = {
                "description": "Billing settled but no ledger debit/credit created",
                "double_entry_violation": True,
            }

        elif failure_type == "currency_conversion_drift":
            currency = str(self._rng.choice(["EUR", "GBP", "JPY"]))
            billing_rate = SUPPORTED_CURRENCIES[currency]
            # Ledger uses a rate from 24 hours later — FX drift
            drift = float(self._rng.uniform(-0.02, 0.02))
            ledger_rate = billing_rate * (1 + drift)
            amount_usd = base_amount
            record["event_id"] = self.generate_event_id()
            record["currency"] = currency
            record["fx_rate"] = billing_rate
            record["amount_usd"] = round(amount_usd, 6)
            record["amount_local"] = round(amount_usd * billing_rate, 6)
            record["user_id"] = f"u_{str(uuid.uuid4())[:12]}"
            record["failure_details"] = {
                "billing_fx_rate": billing_rate,
                "ledger_fx_rate": round(ledger_rate, 6),
                "fx_drift_pct": round(drift * 100, 4),
                "amount_discrepancy_usd": round(
                    amount_usd * abs(ledger_rate - billing_rate), 6
                ),
            }

        elif failure_type == "retroactive_price_change":
            # Campaign pricing changed AFTER these events were billed
            old_price = campaign.bid_price
            new_price = round(old_price * float(self._rng.uniform(0.7, 1.4)), 6)
            record["event_id"] = self.generate_event_id()
            record["amount_usd"] = round(old_price, 6)  # billed at old price
            record["amount_local"] = round(old_price, 6)
            record["user_id"] = f"u_{str(uuid.uuid4())[:12]}"
            record["failure_details"] = {
                "billed_at_price": old_price,
                "correct_price_after_change": new_price,
                "price_delta": round(new_price - old_price, 6),
                "requires_adjustment": True,
                "description": "Campaign price changed retroactively after billing settled",
            }

        self._billing_buffer.append(record)
        return record

    def _get_schema(self) -> Dict:
        return {
            "type": "object",
            "required": ["billing_id", "event_id", "billing_timestamp", "amount_usd"],
        }


class LedgerEntryGenerator(BaseEventGenerator):

    EVENT_TYPE = "ledger_entries"

    def __init__(self, config, shared_state: SharedGeneratorState,
                 billing_generator: BillingRecordGenerator, validator=None):
        edge_injectors = []
        super().__init__(config, shared_state, edge_injectors, validator)
        self._billing_gen = billing_generator
        self._recon_cfg = config.reconciliation

    def _generate_record(self) -> Dict[str, Any]:
        now = self.current_event_time()

        # Source from billing buffer
        if not self._billing_gen._billing_buffer:
            # No billing records yet — generate a standalone ledger entry
            return self._generate_orphan_ledger_entry(now)

        billing = self._billing_gen._billing_buffer[
            int(self._rng.integers(0, len(self._billing_gen._billing_buffer)))
        ]
        return self._generate_ledger_pair(billing, now)

    def _generate_ledger_pair(self, billing: Dict, now: datetime) -> Dict:

        debit_amount = billing.get("amount_usd", 0.0)
        credit_amount = debit_amount

        is_failure = billing.get("is_reconciliation_failure", False)
        failure_type = billing.get("failure_type")

        # Introduce ledger-specific failures
        ledger_error = None
        if is_failure and failure_type in ("ledger_mismatch", "missing_ledger_entry"):
            if failure_type == "missing_ledger_entry":
                # Return a null-like record to signal missing entry
                return {
                    "ledger_id": self.generate_event_id(),
                    "billing_id": billing["billing_id"],
                    "event_id": billing.get("event_id"),
                    "entry_timestamp": self.to_iso(now),
                    "ingestion_timestamp": None,
                    "campaign_id": billing.get("campaign_id"),
                    "advertiser_id": billing.get("advertiser_id"),
                    "debit_account": None,
                    "credit_account": None,
                    "debit_amount_usd": None,
                    "credit_amount_usd": None,
                    "currency": billing.get("currency", "USD"),
                    "is_balanced": False,
                    "ledger_error": "missing_entry",
                    "reconciliation_status": "failed",
                }
            elif failure_type == "ledger_mismatch":
                delta = billing["failure_details"].get("delta", 0.0)
                credit_amount = round(debit_amount + delta, 6)
                ledger_error = "amount_mismatch"

        # Rounding: penny discrepancy (very common in prod)
        if self._rng.random() < 0.005:
            credit_amount = round(credit_amount + 0.01, 6)
            ledger_error = ledger_error or "rounding_discrepancy"

        is_balanced = abs(debit_amount - credit_amount) < 0.001

        return {
            "ledger_id": self.generate_event_id(),
            "billing_id": billing["billing_id"],
            "event_id": billing.get("event_id"),
            "entry_timestamp": self.to_iso(now),
            "ingestion_timestamp": None,
            "campaign_id": billing.get("campaign_id"),
            "advertiser_id": billing.get("advertiser_id"),
            "debit_account": f"AR_{billing.get('advertiser_id', 'unknown')}",
            "credit_account": "REVENUE_DIGITAL_ADV",
            "debit_amount_usd": round(debit_amount, 6),
            "credit_amount_usd": round(credit_amount, 6),
            "currency": billing.get("currency", "USD"),
            "fx_rate": billing.get("fx_rate", 1.0),
            "is_balanced": is_balanced,
            "balance_delta": round(debit_amount - credit_amount, 6),
            "ledger_error": ledger_error,
            "pricing_model": billing.get("pricing_model"),
            "reconciliation_status": "pending" if is_balanced else "failed",
            "is_reconciliation_failure": not is_balanced or billing.get("is_reconciliation_failure", False),
        }

    def _generate_orphan_ledger_entry(self, now: datetime) -> Dict:
        campaign = self.shared_state.campaigns.get_active_campaign()
        amount = round(float(self._rng.lognormal(mean=0.5, sigma=0.8)), 6)
        return {
            "ledger_id": self.generate_event_id(),
            "billing_id": f"ORPHAN_{str(uuid.uuid4())[:12]}",
            "event_id": None,
            "entry_timestamp": self.to_iso(now),
            "ingestion_timestamp": None,
            "campaign_id": campaign.campaign_id,
            "advertiser_id": campaign.advertiser_id,
            "debit_account": f"AR_{campaign.advertiser_id}",
            "credit_account": "REVENUE_DIGITAL_ADV",
            "debit_amount_usd": amount,
            "credit_amount_usd": amount,
            "currency": "USD",
            "fx_rate": 1.0,
            "is_balanced": True,
            "balance_delta": 0.0,
            "ledger_error": "no_billing_reference",
            "reconciliation_status": "failed",
            "is_reconciliation_failure": True,
        }

    def _get_schema(self) -> Dict:
        return {
            "type": "object",
            "required": ["ledger_id", "billing_id", "debit_amount_usd", "credit_amount_usd"],
        }
"""

path = "/mnt/c/Users/JatinJangid/.vscode/adtech-databricks/AdTech - Finance project/Adtech - generators/generator_src/generators/reconciliation_generator.py"

with open(path, "w") as f:
    f.write(code.strip())
print(f"Written: {path}")

Written: /mnt/c/Users/JatinJangid/.vscode/adtech-databricks/AdTech - Finance project/Adtech - generators/generator_src/generators/reconciliation_generator.py


In [20]:
code = """
import logging
import os
import signal
import time
from datetime import datetime, timezone
from typing import Dict, List, Optional
import numpy as np

logger = logging.getLogger(__name__)


class GeneratorOrchestrator:

    def __init__(self, config, validator=None):
        self.config = config
        self._running = False
        self._iteration = 0
        self._total_written = 0
        self._errors = 0
        self._start_time = None

        base_seed = config.platform.seed
        rng = np.random.default_rng(base_seed ^ 0xDEADBEEF)

        # ── Shared State ──────────────────────────────────────────────
        from simulation.shared_state import SharedGeneratorState
        self.shared_state = SharedGeneratorState(config, rng)

        # ── Schema Evolution (cross-cutting) ──────────────────────────
        from edge_cases.schema_evolution_injector import SchemaEvolutionController
        self.schema_controller = SchemaEvolutionController(config, rng)

        # ── Traffic Dynamics (cross-cutting) ──────────────────────────
        from simulation.traffic_dynamics import BurstTrafficController, HotPartitionSimulator
        self.burst_controller = BurstTrafficController(config, rng)
        self.hot_partition_sim = HotPartitionSimulator(config, rng, self.shared_state)

        # ── Replay Manager ────────────────────────────────────────────
        from simulation.replay_manager import ReplayManager
        self.replay_manager = ReplayManager(config)

        if config.replay.enabled:
            self.replay_manager.start_replay(
                replay_type=config.replay.replay_type,
                seed=config.replay.replay_seed,
                reason=config.replay.replay_reason,
            )

        # ── Build Generator Pipeline ──────────────────────────────────
        self._generators = self._build_generators(validator, rng)

        signal.signal(signal.SIGINT, self._handle_shutdown)
        signal.signal(signal.SIGTERM, self._handle_shutdown)

    def _build_generators(self, validator, rng) -> List:
        from generators.bid_request_generator import BidRequestGenerator
        from generators.impression_generator import ImpressionGenerator
        from generators.click_generator import ClickGenerator
        from generators.conversion_generator import ConversionGenerator
        from generators.campaign_cdc_generator import CampaignCDCGenerator
        from generators.refund_generator import RefundGenerator
        from generators.reconciliation_generator import (
            BillingRecordGenerator, LedgerEntryGenerator
        )

        billing_gen = BillingRecordGenerator(self.config, self.shared_state, validator)
        ledger_gen = LedgerEntryGenerator(
            self.config, self.shared_state, billing_gen, validator
        )

        return [
            # Funnel order matters: each layer references the one above
            BidRequestGenerator(self.config, self.shared_state, validator),
            ImpressionGenerator(self.config, self.shared_state, validator),
            ClickGenerator(self.config, self.shared_state, validator),
            ConversionGenerator(self.config, self.shared_state, validator),
            # Financial layer (reference conversions/clicks)
            RefundGenerator(self.config, self.shared_state, validator),
            billing_gen,
            ledger_gen,
            # Metadata stream
            CampaignCDCGenerator(self.config, self.shared_state, validator),
        ]

    def _build_cross_cutting_injectors(self, rng) -> Dict:
        from edge_cases.time_chaos_injector import TimeChaosInjector
        from edge_cases.data_quality_injector import DataQualityInjector
        from edge_cases.schema_evolution_injector import SchemaEvolutionInjector

        return {
            event_type: [
                TimeChaosInjector(self.config, event_type),
                DataQualityInjector(self.config, event_type),
                SchemaEvolutionInjector(
                    self.config, event_type, self.schema_controller
                ),
            ]
            for event_type in [
                "bid_requests", "impressions", "clicks", "conversions",
                "refunds", "billing_records", "ledger_entries", "campaign_cdc",
            ]
        }

    def _handle_shutdown(self, signum, frame):
        logger.info("Shutdown signal received. Finishing current batch...")
        self._running = False

    def run(self, max_iterations: Optional[int] = None):
        self._running = True
        self._start_time = time.monotonic()

        # Build cross-cutting injectors
        cross_injectors = self._build_cross_cutting_injectors(
            np.random.default_rng(self.config.platform.seed ^ 0xCAFEBABE)
        )
        cross_rng = np.random.default_rng(self.config.platform.seed ^ 0xF00D)

        logger.info("=" * 70)
        logger.info("AdTech Data Generator V2 — All 14 FAANG Challenges Active")
        logger.info(f"  Active generators: {[g.EVENT_TYPE for g in self._generators]}")
        logger.info(f"  Schema evolution: {self.config.schema_evolution.enabled}")
        logger.info(f"  Time chaos: {self.config.time_chaos.enabled}")
        logger.info(f"  Burst traffic: {self.config.burst_traffic.enabled}")
        logger.info("=" * 70)

        while self._running:
            if max_iterations is not None and self._iteration >= max_iterations:
                break

            batch_start = time.monotonic()

            # ── Tick cross-cutting controllers ────────────────────────
            burst_mult, burst_cause = self.burst_controller.get_current_multiplier()
            self.burst_controller.maybe_trigger_burst()
            self.hot_partition_sim.tick()

            if burst_cause:
                logger.warning(f"[BURST ACTIVE] {burst_cause} × {burst_mult:.1f}")

            # ── Run each generator ────────────────────────────────────
            batch_written = 0
            for generator in self._generators:
                try:
                    # Run the generator's normal batch
                    records = self._run_generator_batch(generator, burst_mult)

                    # Apply cross-cutting injectors (time chaos, DQ, schema evolution)
                    injectors = cross_injectors.get(generator.EVENT_TYPE, [])
                    for injector in injectors:
                        records = injector.inject(records, cross_rng, generator.metrics)

                    # Apply hot partition forcing
                    records = [
                        self.hot_partition_sim.apply_hot_key(r, cross_rng)
                        for r in records
                    ]

                    # Apply burst metadata tagging
                    if burst_cause:
                        records = [
                            self.burst_controller.inject_burst_metadata(r, burst_cause)
                            for r in records
                        ]

                    # Apply replay annotation
                    if self.config.replay.enabled:
                        records = [self.replay_manager.annotate_record(r) for r in records]

                    # Write final records
                    written = generator._write_batch(records)
                    batch_written += len(records)

                except Exception as e:
                    self._errors += 1
                    logger.error(
                        f"Generator {generator.EVENT_TYPE} failed: {e}",
                        exc_info=True
                    )

            self._total_written += batch_written
            self._iteration += 1

            if self._iteration % 10 == 0:
                self._log_aggregate_stats()

            elapsed = time.monotonic() - batch_start
            sleep_time = max(0, self.config.platform.batch_interval_seconds - elapsed)
            if sleep_time > 0:
                time.sleep(sleep_time)

        if self.config.replay.enabled:
            self.replay_manager.end_replay()

        self._log_final_stats()

    def _run_generator_batch(self, generator, burst_multiplier: float) -> List[Dict]:
        base_batch_size = generator._calculate_batch_size()
        burst_batch_size = max(1, int(base_batch_size * burst_multiplier))

        records = []
        for _ in range(burst_batch_size):
            try:
                record = generator._generate_record()
                record["ingestion_timestamp"] = generator.to_iso(
                    __import__("datetime").datetime.now(
                        __import__("datetime").timezone.utc
                    )
                )
                generator._add_to_dedup_buffer(record["event_id"])
                records.append(record)
                generator.metrics.records_generated += 1
            except Exception as e:
                logger.warning(f"Record generation failed in {generator.EVENT_TYPE}: {e}")

        # Apply per-generator edge case injectors
        records = generator._apply_edge_cases(records)
        return records

    def _log_aggregate_stats(self):
        elapsed = time.monotonic() - self._start_time
        rps = self._total_written / elapsed if elapsed > 0 else 0
        logger.info(
            f"[ORCHESTRATOR] iter={self._iteration} written={self._total_written:,} "
            f"errors={self._errors} rps={rps:.1f} "
            f"schema_versions={self.schema_controller._current_versions}"
        )

    def _log_final_stats(self):
        elapsed = time.monotonic() - self._start_time
        logger.info("=" * 70)
        logger.info("Run Complete")
        logger.info(f"  Iterations:    {self._iteration:,}")
        logger.info(f"  Records:       {self._total_written:,}")
        logger.info(f"  Errors:        {self._errors}")
        logger.info(f"  Throughput:    {self._total_written/max(elapsed,1):.1f} rps")
        logger.info(f"  Schema state:  {self.schema_controller._current_versions}")
        logger.info("=" * 70)
"""

path = "/mnt/c/Users/JatinJangid/.vscode/adtech-databricks/AdTech - Finance project/Adtech - generators/generator_src/orchestrator.py"

with open(path, "w") as f:
    f.write(code.strip())
print(f"Written: {path}")

Written: /mnt/c/Users/JatinJangid/.vscode/adtech-databricks/AdTech - Finance project/Adtech - generators/generator_src/orchestrator.py


In [21]:
code = """
import logging
from datetime import datetime, timezone, timedelta
from typing import Any, Dict, List, Optional, Tuple

logger = logging.getLogger(__name__)

try:
    import jsonschema
    JSONSCHEMA_AVAILABLE = True
except ImportError:
    JSONSCHEMA_AVAILABLE = False
    logger.warning("jsonschema not available; structural validation disabled")


BUSINESS_RULES = {
    "bid_request": [
        lambda r: (r.get("bid_price", 0) >= 0, "bid_price must be non-negative"),
        lambda r: (r.get("floor_price_cpm", 0) >= 0, "floor_price_cpm must be non-negative"),
        lambda r: (r.get("event_id") is not None, "event_id is required"),
    ],
    "impression": [
        lambda r: (r.get("clearing_price", 0) >= 0, "clearing_price must be non-negative"),
        lambda r: (r.get("viewability", 1) <= 1.0, "viewability must be <= 1.0"),
    ],
    "click": [
        lambda r: (r.get("cost", 0) >= 0 or r.get("_financial_edge_case") == "negative_adjustment",
                   "cost must be non-negative unless negative_adjustment"),
        lambda r: (r.get("time_to_click_seconds", 1) >= 0, "time_to_click must be non-negative"),
    ],
    "conversion": [
        lambda r: (r.get("revenue") is not None, "revenue is required"),
        lambda r: (r.get("attribution_window_hours", 1) > 0, "attribution_window must be positive"),
    ],
    "campaign_cdc": [
        lambda r: (r.get("change_type") in
                   ["budget_update", "pricing_model_change", "status_change",
                    "targeting_update", "heartbeat"],
                   "invalid change_type"),
    ],
}

SCHEMAS: Dict[str, Dict] = {
    "bid_request": {
        "type": "object",
        "required": ["event_id", "event_timestamp", "campaign_id", "user_id",
                     "device_type", "bid_price", "pricing_model"],
        "properties": {
            "event_id": {"type": "string", "minLength": 1},
            "bid_price": {"type": "number", "minimum": 0},
            "device_type": {"enum": ["mobile", "desktop", "tablet"]},
            "pricing_model": {"enum": ["CPC", "CPM", "CPA"]},
        },
        "additionalProperties": True,
    },
    "impression": {
        "type": "object",
        "required": ["event_id", "event_timestamp", "campaign_id", "user_id", "clearing_price"],
        "properties": {
            "clearing_price": {"type": "number", "minimum": 0},
            "viewability": {"type": "number", "minimum": 0, "maximum": 1},
        },
        "additionalProperties": True,
    },
    "click": {
        "type": "object",
        "required": ["event_id", "event_timestamp", "campaign_id", "user_id"],
        "additionalProperties": True,
    },
    "conversion": {
        "type": "object",
        "required": ["event_id", "event_timestamp", "campaign_id", "user_id", "revenue"],
        "properties": {
            "revenue": {"type": "number"},
        },
        "additionalProperties": True,
    },
    "campaign_cdc": {
        "type": "object",
        "required": ["event_id", "event_timestamp", "campaign_id", "change_type"],
        "additionalProperties": True,
    },
}


class SchemaValidator:

    def __init__(self, config, max_record_age_hours: int = 72):
        self._config = config
        self._max_age = timedelta(hours=max_record_age_hours)

    def validate(self, record: Dict, event_type: str) -> Tuple[bool, List[str]]:
        errors = []

        # 1. JSON Schema structural validation
        if JSONSCHEMA_AVAILABLE and event_type in SCHEMAS:
            try:
                jsonschema.validate(record, SCHEMAS[event_type])
            except jsonschema.ValidationError as e:
                errors.append(f"schema: {e.message}")
            except jsonschema.SchemaError as e:
                logger.error(f"Invalid schema definition for {event_type}: {e}")

        # 2. Business rule validation
        for rule_fn in BUSINESS_RULES.get(event_type, []):
            try:
                passed, message = rule_fn(record)
                if not passed:
                    errors.append(f"business_rule: {message}")
            except Exception as e:
                errors.append(f"rule_error: {e}")

        # 3. Temporal sanity check (very old events are suspicious unless flagged)
        if "event_timestamp" in record and record.get("_late_by_minutes") is None:
            try:
                ts = datetime.fromisoformat(
                    record["event_timestamp"].replace("Z", "+00:00")
                )
                age = datetime.now(timezone.utc) - ts
                if age > self._max_age:
                    errors.append(f"temporal: event_timestamp too old ({age.total_seconds()/3600:.1f}h)")
            except ValueError:
                errors.append("temporal: invalid event_timestamp format")

        return len(errors) == 0, errors
"""

path = "/mnt/c/Users/JatinJangid/.vscode/adtech-databricks/AdTech - Finance project/Adtech - generators/generator_src/validation/schema_validator.py"

with open(path, "w") as f:
    f.write(code.strip())
print(f"Written: {path}")

Written: /mnt/c/Users/JatinJangid/.vscode/adtech-databricks/AdTech - Finance project/Adtech - generators/generator_src/validation/schema_validator.py


In [22]:
import os

BASE = "/mnt/c/Users/JatinJangid/.vscode/adtech-databricks/AdTech - Finance project/Adtech - generators/generator_src"
REQUIRED = [
    "core/config_loader.py", "core/base_generator.py",
    "simulation/shared_state.py", "simulation/traffic_dynamics.py",
    "simulation/replay_manager.py",
    "edge_cases/injectors.py", "edge_cases/time_chaos_injector.py",
    "edge_cases/schema_evolution_injector.py", "edge_cases/data_quality_injector.py",
    "validation/schema_validator.py",
    "generators/bid_request_generator.py", "generators/impression_generator.py",
    "generators/click_generator.py", "generators/conversion_generator.py",
    "generators/campaign_cdc_generator.py", "generators/refund_generator.py",
    "generators/reconciliation_generator.py", "orchestrator.py",
]

all_ok = True
for rel_path in REQUIRED:
    full = f"{BASE}/{rel_path}"
    exists = os.path.exists(full)
    size = os.path.getsize(full) if exists else 0
    status = "OK" if (exists and size > 100) else "MISSING or EMPTY"
    if "MISSING" in status:
        all_ok = False
    print(f"  [{status}]  {rel_path}")

print("\n✓ All files ready" if all_ok else "\n⚠ Fix missing files before continuing")

  [OK]  core/config_loader.py
  [OK]  core/base_generator.py
  [OK]  simulation/shared_state.py


  [OK]  simulation/traffic_dynamics.py
  [OK]  simulation/replay_manager.py
  [OK]  edge_cases/injectors.py
  [OK]  edge_cases/time_chaos_injector.py


  [OK]  edge_cases/schema_evolution_injector.py
  [OK]  edge_cases/data_quality_injector.py
  [OK]  validation/schema_validator.py
  [OK]  generators/bid_request_generator.py
  [OK]  generators/impression_generator.py
  [OK]  generators/click_generator.py


  [OK]  generators/conversion_generator.py
  [OK]  generators/campaign_cdc_generator.py
  [OK]  generators/refund_generator.py
  [OK]  generators/reconciliation_generator.py
  [OK]  orchestrator.py

✓ All files ready


In [23]:
path = "/mnt/c/Users/JatinJangid/.vscode/adtech-databricks/AdTech - Finance project/Adtech - generators/generator_src/core/base_generator.py"

with open(path, "r") as f:
    content = f.read()

# Don't create a new list if subclass already created one via _recent_event_ids_ref()
old = "self._recent_event_ids: List[str] = []"
new = "if not hasattr(self, '_recent_event_ids'):\n            self._recent_event_ids: List[str] = []"

if old in content:
    content = content.replace(old, new)
    with open(path, "w") as f:
        f.write(content)
    print("base_generator.py patched")
else:
    print("Line not found — check manually")
    # Show surrounding lines to help locate it
    for i, line in enumerate(content.splitlines()):
        if "_recent_event_ids" in line:
            print(f"  line {i+1}: {line}")

base_generator.py patched


In [24]:
import os

BASE = "/mnt/c/Users/JatinJangid/.vscode/adtech-databricks/AdTech - Finance project/Adtech - generators/generator_src/generators"

GENERATOR_FILES = [
    "bid_request_generator.py",
    "impression_generator.py",
    "click_generator.py",
    "conversion_generator.py",
    "campaign_cdc_generator.py",
    "refund_generator.py",
    "reconciliation_generator.py",
]

OLD_METHOD = """    def _recent_event_ids_ref(self):
        # Lazy reference — populated after super().__init__
        return self._recent_event_ids"""

NEW_METHOD = """    def _recent_event_ids_ref(self):
        # Pre-initialize if super().__init__() hasn't run yet
        if not hasattr(self, '_recent_event_ids'):
            self._recent_event_ids = []
        return self._recent_event_ids"""

# Some files may have slightly different comment text — handle both
OLD_METHOD_ALT = """    def _recent_event_ids_ref(self):
        return self._recent_event_ids"""

for fname in GENERATOR_FILES:
    fpath = os.path.join(BASE, fname)
    if not os.path.exists(fpath):
        print(f"  SKIP (not found): {fname}")
        continue

    with open(fpath, "r") as f:
        content = f.read()

    if "_recent_event_ids_ref" not in content:
        print(f"  SKIP (no ref method): {fname}")
        continue

    if OLD_METHOD in content:
        content = content.replace(OLD_METHOD, NEW_METHOD)
        patched = True
    elif OLD_METHOD_ALT in content:
        content = content.replace(OLD_METHOD_ALT, NEW_METHOD)
        patched = True
    else:
        # Regex fallback for any whitespace variation
        import re
        pattern = r'( +)def _recent_event_ids_ref\(self\):\n.*?return self\._recent_event_ids'
        match = re.search(pattern, content)
        if match:
            indent = match.group(1)
            replacement = (
                f"{indent}def _recent_event_ids_ref(self):\n"
                f"{indent}    if not hasattr(self, '_recent_event_ids'):\n"
                f"{indent}        self._recent_event_ids = []\n"
                f"{indent}    return self._recent_event_ids"
            )
            content = content[:match.start()] + replacement + content[match.end():]
            patched = True
        else:
            patched = False

    if patched:
        with open(fpath, "w") as f:
            f.write(content)
        print(f"  PATCHED: {fname}")
    else:
        print(f"  MANUAL FIX NEEDED: {fname}")
        print(f"    Find _recent_event_ids_ref and replace its body with:")
        print(f"        if not hasattr(self, '_recent_event_ids'):")
        print(f"            self._recent_event_ids = []")
        print(f"        return self._recent_event_ids")

  PATCHED: bid_request_generator.py


  PATCHED: impression_generator.py
  PATCHED: click_generator.py
  PATCHED: conversion_generator.py


  SKIP (no ref method): campaign_cdc_generator.py
  SKIP (no ref method): refund_generator.py


  SKIP (no ref method): reconciliation_generator.py


In [25]:
import os

BASE = "/mnt/c/Users/JatinJangid/.vscode/adtech-databricks/AdTech - Finance project/Adtech - generators/generator_src"

# Check base_generator.py
bg_path = f"{BASE}/core/base_generator.py"
with open(bg_path) as f:
    bg = f.read()

bg_ok = "if not hasattr(self, '_recent_event_ids')" in bg
print(f"base_generator.py — hasattr guard: {'OK' if bg_ok else 'MISSING'}")

# Check each generator
gen_dir = f"{BASE}/generators"
for fname in os.listdir(gen_dir):
    if not fname.endswith(".py") or fname == "__init__.py":
        continue
    with open(f"{gen_dir}/{fname}") as f:
        content = f.read()
    if "_recent_event_ids_ref" not in content:
        continue
    ok = "if not hasattr(self, '_recent_event_ids')" in content
    print(f"  {fname:<40} {'OK' if ok else 'NEEDS MANUAL FIX'}")

print("\nAll OK — go re-run Notebook 03 Cell 4" if bg_ok else "\nFix base_generator.py manually")

base_generator.py — hasattr guard: OK


  bid_request_generator.py                 OK


  click_generator.py                       OK
  conversion_generator.py                  OK
  impression_generator.py                  OK



All OK — go re-run Notebook 03 Cell 4
